In [1]:
import os, shutil, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gamma as gamma_dist
from scipy.stats import kstest, gamma as gamma_dist, normaltest, norm

matplotlib.use('Agg')

import copy
try:
    from joblib import Parallel, delayed
    _HAS_JOBLIB = True
except ImportError:
    _HAS_JOBLIB = False

[Top 10 Vehicles in Malaysia](https://data.gov.my/dashboard/car-popularity)

[Guide to calculating insurance premium](https://bengkelbergerak.my/en/blog/kira-insurans-kereta)

[Premium calculator by Carso](https://www.carso.my/tool/car-insurance-calculator)

[Flooding risk weightage](https://www.dosm.gov.my/uploads/content-downloads/file_20220929154540.pdf)

In [2]:
# ============================================================================
# ALL MODELLING ASSUMPTIONS / CONFIG - single source of truth
#   ENGINE_CAPACITY_BANDS : tariff engine bands
#   COHORT_CONFIG         : cohort-generation assumptions
#   DTYPE_DICT            : canonical column schema
#   DRIVER_AGE_LOADING    : rating loading by driver band
#   PERIL_DIST / PERIL_BASE : claim severity model constants
# ============================================================================

ENGINE_CAPACITY_BANDS = [
    "0 to 1,400 cc / EV up to 70 kW",
    "1,401 to 1,650 cc / EV 71 - 100 kW",
    "1,651 - 2,200 cc / EV 101 - 125 kW",
    "2,201 - 3,050 cc / EV 126 - 150 kW",
    "3,051 - 4,100 cc / EV 151 - 200 kW",
    "4,101 - 4,250 cc / EV 201 - 250 kW",
    "4,251 - 4,400 cc / EV 251 - 300 kW",
    "Over 4,400 cc / EV > 300 kW",
]

COHORT_CONFIG = {
    'n': 10000,
    'coverage_pct': {
        'Comprehensive': 0.65,
        'TPFT': 0.20,
        'TPO': 0.15
    },
    'vehicle_pct': {
        'ICE': 0.90,
        'EV': 0.10
    },
    'sa_stats': {
        'ICE': {
            'lambda': 50000,
            'spread': 0.5
        },
        'EV': {
            'lambda': 80000,
            'spread': 0.5
        }
    },
    'region_pct': {
        'Peninsular Malaysia': 0.80,
        'East Malaysia (Sabah, Sawarak & Labuan)': 0.20
    },
    'generation_pct': {
        'Young Adults': 0.40,
        'Adults': 0.40,
        'Mature Adults': 0.15,
        'Seniors': 0.05
    },
    'age_bands': {
        'Young Adults': (18, 28),
        'Adults': (28, 46),
        'Mature Adults': (46, 66),
        'Seniors': (66, 76)
    },
    'gender_pct': {
        'Male': 0.55,
        'Female': 0.45
    },
    'marital_pct': {
        'Young Adults': {'Single': 0.85, 'Married': 0.15},
        'Adults': {'Single': 0.50, 'Married': 0.50},
        'Mature Adults': {'Single': 0.20, 'Married': 0.80},
        'Seniors': {'Single': 0.20, 'Married': 0.80}},
    'car_age_median': {
        'Young Adults': 2.0,
        'Adults': 3.5,
        'Mature Adults': 5.0,
        'Seniors': 5.5
    },
    'car_age_sigma': 1.5,
    'risk_pct': {
        'Peninsular Malaysia': {
            'FLOOD_RISK': [0.60, 0.40],
            'THEFT_RISK': [0.40, 0.60]
        },
        'East Malaysia (Sabah, Sawarak & Labuan)': {
            'FLOOD_RISK': [0.00, 1.00],
            'THEFT_RISK': [0.15, 0.85]
        }
    },
    'ncd_table': {
        0: 0.00,
        1: 0.25,
        2: 0.30,
        3: 0.3833,
        4: 0.45,
        5: 0.55
    },
    'ncd_entry': {
        'years': [0, 1, 2, 3, 4, 5],
        'weights': [0.30, 0.22, 0.16, 0.13, 0.10, 0.09]
    },
    'cohort_year': 2026,
    'claim_frequency_base': -2.00,
    'engine_weights': [0.25, 0.20, 0.18, 0.15, 0.10, 0.07, 0.03, 0.02],
    'seed': 42,
    'entrant_profile': {},
    'ev_share_by_year': {},
    'entrant_annual_growth': 0.00,
    # 'ev_share_by_year': {
    #     2026: 0.10, 2030: 0.12, 2035: 0.25, 2040: 0.40, 2045: 0.55
    # },
    # 'entrant_annual_growth': 0.03,
    'entrant_ncd_zero': False,
}

COHORT_CONFIG['entrant_base_count'] = int(0.50 * COHORT_CONFIG['n'])


def interp_schedule(sched, year):
    """Linear interpolation of a {year: value} schedule (clamped at ends).
    Empty schedule -> 0.0 (no ramp configured)."""
    yrs = sorted(sched)
    if not yrs:
        return 0.0
    if year <= yrs[0]:
        return sched[yrs[0]]
    if year >= yrs[-1]:
        return sched[yrs[-1]]
    for a, b in zip(yrs, yrs[1:]):
        if a <= year <= b:
            t = (year - a) / (b - a)
            return sched[a] + t * (sched[b] - sched[a])

DTYPE_DICT = {
    "POLID": "string",

    # Plan details (required to obtain basic premium)
    "COVERAGE_TYPE": "category",
    "VEHICLE_TYPE": "category",
    "CAR_AGE": "int64",
    "SUM_ASSURED": "float64",
    "REGION": "category",
    "ENGINE_CAPACITY": "category",

    # Insured details
    "DRIVER_AGE_CAT": "category",
    "DRIVER_AGE": "int64",
    "DRIVER_GENDER": "category",
    "MARITAL_STATUS": "category",

    # Refinements
    "FLOOD_RISK": "boolean",
    "THEFT_RISK": "boolean",

    # Output from plan details
    "BASIC_PREMIUM": "float64",
    "FINAL_PREMIUM_SST": "float64",

    # Calculation specific variables
    "NCD_LEVEL": "float64",
    "NCD_YEARS": "int64",
    "COHORT_YEAR": "int64"
}

DRIVER_AGE_LOADING = {
    "Young Adults": 1.20,
    "Adults": 1.05,
    "Mature Adults": 1.00,
    "Seniors": 1.05,
}

PERIL_DIST = {
    'Comprehensive': {
        'AD': 0.58, 'Windscreen': 0.15, 'Theft': 0.08,
        'Fire': 0.04, 'TPPD': 0.12, 'TPBI': 0.03
    },
    'TPO': {
        'TPPD': 0.78, 'TPBI': 0.22
    },
    'TPFT': {
        'TPPD': 0.444, 'TPBI': 0.111,
        'Theft': 0.296, 'Fire': 0.148
    }
}

PERIL_BASE = {
    'TPBI':       {'shape': 0.35, 'scale': 70000, 'cap': float('inf')},
    'TPPD':       {'shape': 0.55, 'scale': 9000,  'cap': 3000000},
    'Windscreen': {'shape': 2.00, 'scale': 700,   'cap': 15000}
}



In [ ]:
# INITIAL COHORT GENERATION - single function, all assumptions in COHORT_CONFIG

def age_band(age):
    """Map an exact driver age to its rating band (band upgrades with age)."""
    if age <= 27:
        return 'Young Adults'
    if age <= 45:
        return 'Adults'
    if age <= 65:
        return 'Mature Adults'
    return 'Seniors'


def _p(weights):
    """Normalise weights to sum exactly to 1 (float-safe for np.random.choice)."""
    a = np.array(list(weights), dtype=float)
    return a / a.sum()


def generate_dataset(cfg, seed=None, n=None, cohort_year=None, polid_prefix='INIT'):
    """Generate a complete policy book from COHORT_CONFIG assumptions.

    Args:
        cfg: dict (COHORT_CONFIG) with all settings/assumptions
        seed: RNG seed (defaults to cfg['seed'])
        n: number of policies (defaults to cfg['n'])
        cohort_year: policy base year (defaults to cfg['cohort_year'])
        polid_prefix: 'INIT' for the initial book, 'ENT' for new entrants

    Returns:
        pd.DataFrame with all policy attributes + premium columns
    """
    n = int(n if n is not None else cfg['n'])
    rng = np.random.default_rng(int(seed if seed is not None else cfg['seed']))
    base_year = int(cohort_year if cohort_year is not None else cfg['cohort_year'])

    df = pd.DataFrame(index=range(n))

    # Coverage / vehicle / region
    df['COVERAGE_TYPE'] = rng.choice(
        list(cfg['coverage_pct']),
        size=n,
        p=_p(cfg['coverage_pct'].values())
    )
    df['VEHICLE_TYPE'] = rng.choice(
        list(cfg['vehicle_pct']),
        size=n,
        p=_p(cfg['vehicle_pct'].values())
    )
    df['REGION'] = rng.choice(
        list(cfg['region_pct']),
        size=n,
        p=_p(cfg['region_pct'].values())
    )

    # Sum assured: log-normal per vehicle type, rounded to RM 1,000
    sa = np.zeros(n)
    for vt, stats in cfg['sa_stats'].items():
        m = df['VEHICLE_TYPE'].values == vt
        sa[m] = rng.lognormal(
            mean=np.log(stats['lambda']),
            sigma=stats['spread'],
            size=int(m.sum())
        )
    df['SUM_ASSURED'] = np.round(sa / 1000) * 1000

    # Engine capacity: fixed hand-set mix (small cars dominant)
    df['ENGINE_CAPACITY'] = rng.choice(
        ENGINE_CAPACITY_BANDS,
        size=n,
        p=_p(cfg['engine_weights'])
    )

    # Driver profile: category by weights, age drawn from category band
    df['DRIVER_AGE_CAT'] = rng.choice(
        list(cfg['generation_pct']),
        size=n,
        p=_p(cfg['generation_pct'].values())
    )
    cats = df['DRIVER_AGE_CAT'].values
    lo = np.array([cfg['age_bands'][c][0] for c in cats])
    hi = np.array([cfg['age_bands'][c][1] for c in cats])
    df['DRIVER_AGE'] = rng.integers(lo, hi, size=n)
    df['DRIVER_GENDER'] = rng.choice(
        list(cfg['gender_pct']),
        size=n,
        p=_p(cfg['gender_pct'].values())
    )
    marital = np.empty(n, dtype=object)
    for cat, probs in cfg['marital_pct'].items():
        m = cats == cat
        marital[m] = rng.choice(
            list(probs),
            size=int(m.sum()),
            p=_p(probs.values())
        )
    df['MARITAL_STATUS'] = marital

    # Vehicle age at inception, capped 0-10
    car_age = np.zeros(n)
    for cat, med in cfg['car_age_median'].items():
        m = cats == cat
        car_age[m] = np.clip(
            np.round(med + rng.normal(0, cfg['car_age_sigma'], size=int(m.sum()))),
            0, 10
        )
    df['CAR_AGE'] = car_age.astype(int)

    # Region flood/theft risk flags
    for col in ('FLOOD_RISK', 'THEFT_RISK'):
        out = np.zeros(n, dtype=bool)
        for region, probs in cfg['risk_pct'].items():
            m = df['REGION'].values == region
            out[m] = rng.choice([True, False], size=int(m.sum()), p=_p(probs[col]))
        df[col] = out

    # NCD entry mix
    df['NCD_YEARS'] = rng.choice(
        cfg['ncd_entry']['years'],
        size=n,
        p=_p(cfg['ncd_entry']['weights'])
    )
    df['NCD_LEVEL'] = df['NCD_YEARS'].apply(lambda y: cfg['ncd_table'].get(int(min(y, 5)), 0.55))
    df['COHORT_YEAR'] = base_year

    # POLID: deterministic prefix-year-sequence (INIT/ENT)
    df['POLID'] = [f"{polid_prefix}{base_year}-{i + 1:06d}" for i in range(n)]

    # Premium: BASIC from tariff, TOTAL_LOADING, FINAL with SST
    df['BASIC_PREMIUM'] = df.apply(calculate_premium, axis=1)
    df['TOTAL_LOADING'] = df.apply(
        lambda r: total_loading(r['DRIVER_AGE_CAT'], r['CAR_AGE']), axis=1
    )
    df['FINAL_PREMIUM_SST'] = df.apply(compute_final_premium, axis=1)

    # Apply the canonical schema dtypes
    for col, dt in DTYPE_DICT.items():
        if col in df.columns:
            df[col] = df[col].astype(dt)
    return df


In [ ]:
# BASIC PREMIUM - Schedule of Motor Tariff 2015 (Form 5 Mathematics Ch.3)
# Graduated tariff:
#   Comprehensive = first-RM1,000 rate + PER_EXTRA x ceil((SA-1000)/1000)
#   TPFT (Third Party, Fire & Theft) = 0.75 x Comprehensive basic (Example 3)
#   TPO = flat tariff rate (no sum-assured scaling)

PER_EXTRA = {
    'Peninsular Malaysia': 26.00,
    'East Malaysia (Sabah, Sawarak & Labuan)': 20.30,
}

MOTOR_TARIFF = {
    'Peninsular Malaysia': {
        'Comprehensive': {
            '0 to 1,400 cc / EV up to 70 kW': 273.80,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 305.50,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 339.10,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 372.60,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 404.30,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 436.00,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 469.60,
            'Over 4,400 cc / EV > 300 kW': 501.30,
        },
        'TPO': {
            '0 to 1,400 cc / EV up to 70 kW': 120.60,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 135.00,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 151.20,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 167.40,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 181.80,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 196.20,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 212.40,
            'Over 4,400 cc / EV > 300 kW': 226.80,
        },
    },
    'East Malaysia (Sabah, Sawarak & Labuan)': {
        'Comprehensive': {
            '0 to 1,400 cc / EV up to 70 kW': 196.20,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 220.00,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 243.90,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 266.50,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 290.40,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 313.00,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 336.90,
            'Over 4,400 cc / EV > 300 kW': 359.50,
        },
        'TPO': {
            '0 to 1,400 cc / EV up to 70 kW': 67.50,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 75.60,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 85.20,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 93.60,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 101.70,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 110.10,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 118.20,
            'Over 4,400 cc / EV > 300 kW': 126.60,
        },
    },
}


def comprehensive_basic(row):
    first = MOTOR_TARIFF[row['REGION']]['Comprehensive'][row['ENGINE_CAPACITY']]
    units = int(np.ceil(max(0.0, (row['SUM_ASSURED'] - 1000.0) / 1000.0)))
    return first + PER_EXTRA[row['REGION']] * units


def calculate_premium(row):
    """Basic premium per Schedule of Motor Tariff 2015 (graduated)."""
    coverage = row['COVERAGE_TYPE']
    if coverage == 'Comprehensive':
        basic = comprehensive_basic(row)
    elif coverage == 'TPFT':
        basic = round(0.75 * comprehensive_basic(row) + 1e-9, 2)
    else:  # TPO - flat tariff rate
        basic = MOTOR_TARIFF[row['REGION']]['TPO'][row['ENGINE_CAPACITY']]
    return round(float(basic), 2)

In [ ]:
# Rating loadings: driver age category + vehicle age
# Driver loading: Youngsters highest (inexperience); experienced cohorts lower.
# Car loading: increases linearly with vehicle age (older car = higher risk).

def driver_age_loading(age_cat):
    return DRIVER_AGE_LOADING.get(age_cat, 1.00)

def car_age_loading(car_age):
    """Linear vehicle-age loading; car age capped at 10 years."""
    return 1 + 0.03 * min(int(car_age), 10)

def total_loading(age_cat, car_age):
    """Combined driver x vehicle loading applied to the premium."""
    return driver_age_loading(age_cat) * car_age_loading(car_age)


In [ ]:
# FINAL_PREMIUM_SST: BASIC x driver/car loading x (1 - NCD) x 1.1^risk flags + 8% SST
# NCD discount applies to ALL coverages (both Comprehensive and TPO).
SST_RATE = 0.08  # Changeable variable - current SST rate in Malaysia

def compute_final_premium(row, sst_rate=SST_RATE):
    """Compute final premium with rating loadings, NCD discount, risk multipliers, SST."""
    loading = total_loading(row['DRIVER_AGE_CAT'], row['CAR_AGE'])
    ncd_discount = 1 - row.get('NCD_LEVEL', 0.0)
    risk_multiplier = 1.1 ** (int(row['FLOOD_RISK']) + int(row['THEFT_RISK']))
    final = row['BASIC_PREMIUM'] * loading * ncd_discount * risk_multiplier * (1 + sst_rate)
    return round(float(final), 2)

# FINAL_PREMIUM_SST / TOTAL_LOADING applied inside generate_dataset (cell 050b235c)


In [7]:
# Build the initial cohort (single generator call)
df = generate_dataset(COHORT_CONFIG, seed=COHORT_CONFIG['seed'])

In [8]:

# Claim Frequency Model (Poisson GLM, log-linear)
# lambda = exp(log_lambda) * coverage_multiplier
# Base exp(-2.00) ~ 0.135 claims/year - realistic Malaysian market level
# (tweak for realistic LR)

CLAIM_FREQUENCY_BASE = COHORT_CONFIG['claim_frequency_base']

def compute_claim_lambda(row):
    """Compute Poisson rate lambda via log-linear rating model."""
    log_lambda = CLAIM_FREQUENCY_BASE

    cat = row['DRIVER_AGE_CAT']
    if cat == 'Young Adults':
        log_lambda += 0.40        # young drivers: higher risk
    elif cat == 'Seniors':
        log_lambda += 0.26        # seniors: moderate increase

    # Young male interaction
    if cat == 'Young Adults' and row['DRIVER_GENDER'] == 'Male':
        log_lambda += 0.05

    # EV proxy (higher power/repair exposure)
    if row['VEHICLE_TYPE'] == 'EV':
        log_lambda += 0.05

    # Risk flags
    if row['FLOOD_RISK']:
        log_lambda += 0.20
    if row['THEFT_RISK']:
        log_lambda += 0.10

    # Vehicle age: older cars carry higher breakdown/repair frequency
    log_lambda += 0.03 * row['CAR_AGE']

    # NCD safety credit: claim-free drivers are safer
    log_lambda -= 0.05 * row['NCD_YEARS']

    freq = np.exp(log_lambda)

    # Coverage multiplier: TPO has no own-damage exposure
    # Coverage multiplier: TPO no own-damage; TPFT fire/theft only (0.60)
    mult = 0.45 if row['COVERAGE_TYPE'] == 'TPO' else (0.60 if row['COVERAGE_TYPE'] == 'TPFT' else 1.00)
    return freq * mult


df['CLAIM_LAMBDA'] = df.apply(compute_claim_lambda, axis=1)

print('Claim frequency model (Poisson GLM) applied')
print(f"Mean lambda: {df['CLAIM_LAMBDA'].mean():.4f}")
print(f"Min lambda: {df['CLAIM_LAMBDA'].min():.4f}, "
      f"Max lambda: {df['CLAIM_LAMBDA'].max():.4f}")
print(f"TPO mean lambda: {df.loc[df['COVERAGE_TYPE']=='TPO', 'CLAIM_LAMBDA'].mean():.4f}")
print(f"Comp mean lambda: {df.loc[df['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_LAMBDA'].mean():.4f}")
print(f"TPFT mean lambda: {df.loc[df['COVERAGE_TYPE']=='TPFT', 'CLAIM_LAMBDA'].mean():.4f}")


Claim frequency model (Poisson GLM) applied
Mean lambda: 0.1622
Min lambda: 0.0474, Max lambda: 0.3430
TPO mean lambda: 0.0867
Comp mean lambda: 0.1931
TPFT mean lambda: 0.1161


In [9]:
# Claim Severity Model: Per-Peril Gamma with Policy Caps
# Peril mix follows Malaysian retail product structure:
#   Comprehensive: own damage (AD/Windscreen/Theft/Fire) + third party (TPPD/TPBI)
#   TPO: third party only (TPPD/TPBI) - structurally cheaper claims

def sample_claim_peril(coverage_type):
    """Sample a claim peril from the product-specific mix."""
    mix = PERIL_DIST.get(coverage_type, PERIL_DIST['Comprehensive'])
    probs = np.array(list(mix.values()))
    return np.random.choice(list(mix.keys()), p=probs / probs.sum())


def generate_single_claim(coverage_type, sum_assured):
    """Draw one claim amount (RM) with peril-specific Gamma + cap."""
    peril = sample_claim_peril(coverage_type)

    if peril == 'Theft':
        shape, scale = 1.10, max(8000, min(20000, sum_assured * 0.20))
        cap = sum_assured
    elif peril == 'Fire':
        shape, scale = 0.90, max(7000, min(18000, sum_assured * 0.15))
        cap = sum_assured
    elif peril == 'AD':
        shape, scale = 0.60, max(4500, min(12000, sum_assured * 0.10))
        cap = sum_assured
    else:
        spec = PERIL_BASE[peril]
        shape, scale, cap = spec['shape'], spec['scale'], spec['cap']

    amount = np.random.gamma(shape, scale)
    return min(amount, cap), peril


def generate_claim_total(coverage_type, sum_assured, n_claims):
    """Aggregate severity across all claims in a policy-year."""
    if n_claims <= 0:
        return 0.0, ''
    total = 0.0
    perils = []
    for _ in range(int(n_claims)):
        amt, peril = generate_single_claim(coverage_type, sum_assured)
        total += amt
        perils.append(peril)
    return round(total, 2), '/'.join(perils)


print('Per-peril severity model ready:')
print('  Comprehensive perils:', list(PERIL_DIST['Comprehensive'].keys()))
print('  TPO perils:          ', list(PERIL_DIST['TPO'].keys()))
print('  TPFT perils:         ', list(PERIL_DIST['TPFT'].keys()))


Per-peril severity model ready:
  Comprehensive perils: ['AD', 'Windscreen', 'Theft', 'Fire', 'TPPD', 'TPBI']
  TPO perils:           ['TPPD', 'TPBI']
  TPFT perils:          ['TPPD', 'TPBI', 'Theft', 'Fire']


In [10]:

# Retention Model (Binomial Logit Proxy)
# Probability of renewing policy next year.
# PREMIUM_CHANGE_PCT is fed from the annual portfolio trend (see cohort-simulation).
# Note: FINAL_PREMIUM_SST now evolves yearly (loadings + NCD), but the retention
# signal remains the portfolio-level trend to keep retention behavior stable.

def compute_retention_probability(row, premium_change_pct):
    """Compute probability of renewal.

    Key drivers (priority):
    1. Premium increase (highest sensitivity)
    2. Claim occurrence
    3. NCD level (incentive to stay)
    """
    p = 0.80  # Base renewal rate

    # Premium increase sensitivity (highest priority)
    if premium_change_pct > 0.15:
        p -= 0.15
    elif premium_change_pct > 0.05:
        p -= 0.10
    elif premium_change_pct < -0.05:
        p += 0.05  # Discounts improve retention

    # Claim occurrence effect
    p -= 0.25 if row.get('CLAIM_OCCURRED', False) else 0

    # NCD incentive to stay
    if row['NCD_YEARS'] >= 3:
        p += 0.15
    elif row['NCD_YEARS'] >= 2:
        p += 0.08

    return np.clip(p, 0.1, 0.95)


In [11]:
# ============================================================================
# VECTORIZED SIMULATION HELPERS (NumPy) - hot-path replacement for row-wise apply
# Same models as the scalar versions (claim-model-lambda / claim-model-severity /
# sst-premium / retention-model). cfg param threads scenario overrides through.
# ============================================================================

def age_band_array(ages):
    """Vectorized band upgrade: crossing 27/45/65 moves to the next rating band."""
    return np.select(
        [ages <= 27, ages <= 45, ages <= 65],
        ['Young Adults', 'Adults', 'Mature Adults'],
        default='Seniors')


def claim_lambda_array(df, cfg=COHORT_CONFIG):
    """Vectorized Poisson frequency (log-linear GLM), same model as compute_claim_lambda."""
    cat = df['DRIVER_AGE_CAT'].values
    log_l = np.full(len(df), cfg['claim_frequency_base'])
    log_l += 0.40 * (cat == 'Young Adults')
    log_l += 0.26 * (cat == 'Seniors')
    log_l += 0.05 * ((cat == 'Young Adults') & (df['DRIVER_GENDER'].values == 'Male'))
    log_l += 0.05 * (df['VEHICLE_TYPE'].values == 'EV')
    log_l += 0.20 * df['FLOOD_RISK'].values
    log_l += 0.10 * df['THEFT_RISK'].values
    log_l += 0.03 * df['CAR_AGE'].values
    log_l -= 0.05 * df['NCD_YEARS'].values
    cov = df['COVERAGE_TYPE'].values
    mult = np.where(cov == 'TPO', 0.45, np.where(cov == 'TPFT', 0.60, 1.00))
    return np.exp(log_l) * mult


def total_loading_array(df, cfg=COHORT_CONFIG):
    """Vectorized combined driver x vehicle loading."""
    dl_map = cfg.get('driver_age_loading', DRIVER_AGE_LOADING)
    dl = df['DRIVER_AGE_CAT'].map(dl_map).fillna(1.00).values
    cl = 1 + 0.03 * np.minimum(df['CAR_AGE'].values, 10)
    return dl * cl


def final_premium_array(df, cfg=COHORT_CONFIG, sst_rate=SST_RATE):
    """Vectorized FINAL_PREMIUM_SST: BASIC x loading x (1-NCD) x 1.1^flags x (1+SST)."""
    loading = total_loading_array(df, cfg)
    ncd = 1 - df['NCD_LEVEL'].values
    risk = 1.1 ** (df['FLOOD_RISK'].values.astype(int) + df['THEFT_RISK'].values.astype(int))
    return (df['BASIC_PREMIUM'].values * loading * ncd * risk * (1 + sst_rate)).round(2)


def retention_prob_array(df, premium_change_pct):
    """Vectorized binomial-logit retention proxy, same model as compute_retention_probability."""
    p = np.full(len(df), 0.80)
    pc = np.full(len(df), premium_change_pct)
    p -= np.where(pc > 0.15, 0.15, np.where(pc > 0.05, 0.10, 0.0))
    p += np.where(pc < -0.05, 0.05, 0.0)
    p -= 0.25 * df['CLAIM_OCCURRED'].values.astype(float)
    ncd = df['NCD_YEARS'].values
    p += np.select([ncd >= 3, ncd >= 2], [0.15, 0.08], default=0.0)
    return np.clip(p, 0.1, 0.95)


def ncd_level_array(yrs, cfg=COHORT_CONFIG):
    """Vectorized NCD discount lookup (table keyed 0-5; 6+ -> default 0.55)."""
    y = np.asarray(yrs, dtype=int)
    max_y = int(y.max()) if len(y) else 0
    lut = np.array([cfg['ncd_table'].get(min(i, 6), 0.55) for i in range(max_y + 1)])
    return lut[np.clip(y, 0, max_y)]


_PERIL_NAMES = ['AD', 'Windscreen', 'Theft', 'Fire', 'TPPD', 'TPBI']


def _draw_perils(cov, n, rng, cfg):
    """Vectorized peril draw from coverage-specific mixes (cumsum + uniform)."""
    peril_dist = cfg.get('peril_dist', PERIL_DIST)
    P = np.array([[peril_dist[c].get(p, 0.0) for p in _PERIL_NAMES] for c in cov])
    u = rng.random(n)[:, None]
    idx = np.minimum((u > np.cumsum(P, axis=1)).sum(axis=1), len(_PERIL_NAMES) - 1)
    return np.array(_PERIL_NAMES)[idx]


def _peril_params(perils, sa, cfg):
    """Vectorized Gamma shape/scale/cap per peril (caps vs sum assured)."""
    peril_base = cfg.get('peril_base', PERIL_BASE)
    n = len(perils)
    shape = np.zeros(n)
    scale = np.zeros(n)
    cap = np.full(n, np.inf)
    for name in ('TPBI', 'TPPD', 'Windscreen'):
        m = perils == name
        if m.any():
            spec = peril_base[name]
            shape[m] = spec['shape']
            scale[m] = spec['scale']
            cap[m] = spec['cap']
    for name, s_lo, s_hi, s_f, sh in (('Theft', 8000, 20000, 0.20, 1.10),
                                      ('Fire', 7000, 18000, 0.15, 0.90),
                                      ('AD', 4500, 12000, 0.10, 0.60)):
        m = perils == name
        if m.any():
            shape[m] = sh
            scale[m] = np.clip(sa[m] * s_f, s_lo, s_hi)
            cap[m] = sa[m]
    return shape, scale, cap


def simulate_claim_severity_vectorized(df, rng, cfg=COHORT_CONFIG):
    """Vectorized severity: explode by claim count -> peril -> Gamma -> cap -> aggregate."""
    n = len(df)
    amount = np.zeros(n)
    peril_out = np.empty(n, dtype=object)
    peril_out[:] = ''
    counts = df['CLAIM_COUNT'].values
    claim_mask = counts > 0
    if not claim_mask.any():
        return amount, peril_out
    idx = np.repeat(np.flatnonzero(claim_mask), counts[claim_mask])
    cov = df['COVERAGE_TYPE'].values[idx]
    sa = df['SUM_ASSURED'].values[idx]
    perils = _draw_perils(cov, len(idx), rng, cfg)
    shape, scale, cap = _peril_params(perils, sa, cfg)
    amts = np.minimum(rng.gamma(shape, scale), cap)
    np.add.at(amount, idx, amts)
    s = pd.Series(perils).groupby(pd.Series(idx)).agg('/'.join)
    peril_out[np.flatnonzero(claim_mask)] = s.values
    return amount, peril_out


In [12]:
# ============================================================================
# COHORT EVOLUTION SIMULATION (vectorized; local RNG; config-threaded)
# In-force policies age each year (DRIVER_AGE, CAR_AGE +1); claim frequency and
# final premium are recomputed annually with the new ages and NCD (one-year lag:
# year N is priced with the NCD earned through year N-1).
# ============================================================================

def _entrant_vehicle_pct(cfg, year):
    """Vehicle mix for entrants in `year`: base mix with EV share ramped by
    schedule; empty/missing schedule -> base mix unchanged (no custom EV share)."""
    base = dict(cfg['vehicle_pct'])
    sched = cfg.get('ev_share_by_year') or {}
    if not sched:
        return base
    ev = interp_schedule(sched, year)
    if ev >= 1.0:
        return {'EV': 1.0}
    others = [k for k in base if k != 'EV']
    rest = sum(base[k] for k in others)
    out = {k: (1 - ev) * base[k] / rest if rest > 0 else 0.0 for k in others}
    out['EV'] = ev
    return out


def _entrant_count(cfg, year_offset):
    """Dynamic entrant count: base * (1 + growth) ** year_offset (compounding)."""
    return max(0, int(round(cfg['entrant_base_count'] *
                            (1 + cfg['entrant_annual_growth']) ** year_offset)))


def simulate_cohort(df_initial, n_years=5, new_entrants_per_year=None,
                    premium_trend_annual=1.06, seed=42, cfg=COHORT_CONFIG,
                    verbose=True):
    """Simulate cohort evolution (vectorized hot path; per-seed local RNG).

    Args:
        df_initial: Starting cohort (Year 1)
        n_years: Number of years to simulate
        new_entrants_per_year: static entrants/year (None -> dynamic growth-based count)
        premium_trend_annual: Annual premium inflation used for retention only
        seed: Random seed for reproducibility (local Generator)
        cfg: config dict (default COHORT_CONFIG); threaded to helpers + entrants
        verbose: per-year + summary prints

    Returns:
        pd.DataFrame with all policy-year records
    """
    t0 = time.time()
    rng = np.random.default_rng(seed)
    history = []
    df_active = df_initial.copy()

    for year_offset in range(n_years):
        year = cfg['cohort_year'] + year_offset
        df_active['SIM_YEAR'] = year

        # Age in-force policies (brand-new entrants keep fresh ages).
        aging_mask = df_active['COHORT_YEAR'] < year
        if aging_mask.any():
            df_active.loc[aging_mask, 'DRIVER_AGE'] += 1
            df_active.loc[aging_mask, 'CAR_AGE'] = np.minimum(
                df_active.loc[aging_mask, 'CAR_AGE'] + 1, 10
            )
            # Band upgrades with age: crossing 27/45/65 moves to the next rating band
            df_active.loc[aging_mask, 'DRIVER_AGE_CAT'] = age_band_array(
                df_active.loc[aging_mask, 'DRIVER_AGE'].values
            )

        # Recompute frequency + premium with current ages and NCD
        # (NCD_LEVEL here still reflects claims through the PRIOR year -> one-year lag)
        df_active['CLAIM_LAMBDA'] = claim_lambda_array(df_active, cfg)
        df_active['TOTAL_LOADING'] = total_loading_array(df_active, cfg)
        df_active['NCD_LEVEL_PRICED'] = df_active['NCD_LEVEL']
        df_active['FINAL_PREMIUM_SST'] = final_premium_array(df_active, cfg)

        # Simulate claims (Poisson frequency, vectorized)
        df_active['CLAIM_COUNT'] = rng.poisson(df_active['CLAIM_LAMBDA'].values)
        df_active['CLAIM_OCCURRED'] = df_active['CLAIM_COUNT'] > 0

        # Simulate severity (per-peril Gamma, vectorized, aggregated per policy-year)
        amounts, perils = simulate_claim_severity_vectorized(df_active, rng, cfg)
        df_active['CLAIM_AMOUNT'] = amounts
        df_active['CLAIM_PERIL'] = perils

        # Premium change signal (retention only - not stored premium)
        premium_change_pct = premium_trend_annual ** year_offset - 1
        df_active['PREMIUM_CHANGE_PCT'] = premium_change_pct

        # Retention + renewals (vectorized)
        df_active['RENEWAL_PROB'] = retention_prob_array(df_active, premium_change_pct)
        df_active['RENEWED'] = rng.random(len(df_active)) < df_active['RENEWAL_PROB'].values

        # Update NCD based on claims
        df_active.loc[~df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] += 1
        df_active.loc[df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] = 0
        df_active['NCD_LEVEL'] = ncd_level_array(df_active['NCD_YEARS'].values, cfg)

        # Record full year state (including lapsers) for retention analysis
        cols_to_keep = ['POLID', 'COVERAGE_TYPE', 'SUM_ASSURED', 'REGION',
                        'VEHICLE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
                        'CAR_AGE', 'DRIVER_GENDER', 'FLOOD_RISK', 'THEFT_RISK',
                        'BASIC_PREMIUM', 'FINAL_PREMIUM_SST', 'TOTAL_LOADING',
                        'NCD_LEVEL_PRICED', 'NCD_LEVEL',
                        'NCD_YEARS', 'CLAIM_LAMBDA',
                        'SIM_YEAR', 'CLAIM_COUNT', 'CLAIM_OCCURRED', 'CLAIM_AMOUNT',
                        'CLAIM_PERIL', 'PREMIUM_CHANGE_PCT',
                        'RENEWAL_PROB', 'RENEWED', 'COHORT_YEAR']
        history.append(df_active[cols_to_keep].copy())

        if verbose:
            print(f"Year {year}: {len(df_active)} active policies, "
                  f"claims: {df_active['CLAIM_COUNT'].sum()}, "
                  f"freq: {df_active['CLAIM_OCCURRED'].mean():.1%}, "
                  f"avg NCD priced: {df_active['NCD_LEVEL_PRICED'].mean():.2%}, "
                  f"avg premium: RM{df_active['FINAL_PREMIUM_SST'].mean():.2f}, "
                  f"retention: {df_active['RENEWED'].mean():.1%}")

        # New entrants: dynamic profile (EV share ramps by year) + dynamic count
        # (base * (1 + growth)**year_offset) unless a static override is passed.
        if year_offset < n_years - 1:
            n_ent = (new_entrants_per_year if new_entrants_per_year is not None
                     else _entrant_count(cfg, year_offset))
            if n_ent > 0:
                ecfg = copy.deepcopy(cfg)
                for k, v in cfg.get('entrant_profile', {}).items():
                    ecfg[k] = v
                ecfg['vehicle_pct'] = _entrant_vehicle_pct(cfg, year + 1)
                new_cohort = generate_dataset(
                    ecfg, seed=seed + year_offset,
                    n=n_ent, cohort_year=year + 1,
                    polid_prefix='ENT'
                )
                if cfg.get('entrant_ncd_zero', False):
                    new_cohort['NCD_YEARS'] = 0
                    new_cohort['NCD_LEVEL'] = cfg['ncd_table'].get(0, 0.55)
                df_active = pd.concat(
                    [df_active[df_active['RENEWED']], new_cohort],
                    ignore_index=True
                )
            else:
                df_active = df_active[df_active['RENEWED']].copy()
        else:
            df_active = df_active[df_active['RENEWED']].copy()

    cohort_results = pd.concat(history, ignore_index=True)
    if verbose:
        print(f"\nSimulation complete. Total records: {len(cohort_results)}")
        print(f"Year range: {cohort_results['SIM_YEAR'].min()} - {cohort_results['SIM_YEAR'].max()}")
        print(f"Simulation wall time: {time.time() - t0:.1f}s")
    return cohort_results


# Run simulation (single trajectory, full detail - feeds all downstream cells)
cohort_results = simulate_cohort(
    df, n_years=20, new_entrants_per_year=None)


Year 2026: 10000 active policies, claims: 1581, freq: 14.5%, avg NCD priced: 24.79%, avg premium: RM1553.14, retention: 82.0%
Year 2027: 13199 active policies, claims: 2105, freq: 14.5%, avg NCD priced: 30.46%, avg premium: RM1458.48, retention: 74.1%
Year 2028: 14774 active policies, claims: 2297, freq: 14.1%, avg NCD priced: 32.75%, avg premium: RM1416.23, retention: 75.2%
Year 2029: 16113 active policies, claims: 2483, freq: 14.1%, avg NCD priced: 33.94%, avg premium: RM1398.23, retention: 70.7%
Year 2030: 16385 active policies, claims: 2575, freq: 14.4%, avg NCD priced: 34.82%, avg premium: RM1385.39, retention: 70.7%
Year 2031: 16579 active policies, claims: 2485, freq: 13.6%, avg NCD priced: 35.00%, avg premium: RM1368.63, retention: 70.9%
Year 2032: 16750 active policies, claims: 2631, freq: 14.3%, avg NCD priced: 35.64%, avg premium: RM1356.01, retention: 71.3%
Year 2033: 16939 active policies, claims: 2492, freq: 13.5%, avg NCD priced: 35.71%, avg premium: RM1356.06, retention

In [13]:
# ============================================================================
# MONTE CARLO API - scenario x seed grid, summary metrics + percentiles
# Vectorized engine + per-seed local RNG -> safe for parallel seeds.
# ============================================================================

def deep_update(base, overrides):
    """Deep-copy base and recursively merge overrides (None-safe)."""
    out = copy.deepcopy(base)
    if not overrides:
        return out
    for k, v in overrides.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_update(out[k], v)
        else:
            out[k] = v
    return out


def summarize(results_df):
    """One trajectory -> scalar metrics + per-year loss-ratio series."""
    premium = results_df['FINAL_PREMIUM_SST'].sum()
    claims = results_df['CLAIM_AMOUNT'].sum()
    metrics = {
        'overall_lr': claims / premium if premium > 0 else np.nan,
        'claim_freq': results_df['CLAIM_OCCURRED'].mean(),
        'avg_premium': results_df['FINAL_PREMIUM_SST'].mean(),
        'retention': results_df['RENEWED'].mean(),
        'n_policy_years': len(results_df),
    }

    def _yr_lr(g):
        p = g['FINAL_PREMIUM_SST'].sum()
        return g['CLAIM_AMOUNT'].sum() / p if p > 0 else np.nan

    yearly = results_df.groupby('SIM_YEAR').apply(_yr_lr)
    return metrics, yearly


def _run_one(cfg, seed, n_years, new_entrants_per_year,
             book_seed=COHORT_CONFIG['seed']):
    df0 = generate_dataset(cfg, seed=book_seed)
    res = simulate_cohort(df0, n_years=n_years,
                          new_entrants_per_year=new_entrants_per_year,
                          seed=seed, cfg=cfg, verbose=False)
    return summarize(res)


def run_monte_carlo(overrides=None, seeds=(42,), n_years=20,
                    new_entrants_per_year=None, n_jobs=4, book_seed=None):
    """Run a (scenario x seed) Monte Carlo grid.

    Args:
        overrides: None | dict | list[dict] - assumption patches on COHORT_CONFIG
        seeds: int | list[int] - independent replications per scenario
        n_years: simulation horizon
        new_entrants_per_year: static entrants/year override (None -> dynamic
                                growth-based count from cfg)
        n_jobs: parallel workers (threading backend; numpy releases the GIL)
        book_seed: initial-cohort RNG seed (default COHORT_CONFIG['seed']).
                   The MC seed perturbs only the forward path (claims / severity /
                   retention / entrants); the starting book is fixed per book_seed.

    NOTE: the initial book is regenerated from book_seed + the CURRENT COHORT_CONFIG.
    After tweaking assumptions, re-run the book-generation cell so the standalone
    cohort_results book matches what run_monte_carlo builds.

    Returns:
        (metrics_df, summary_df, yearly_lr_df)
          metrics_df:  one row per (scenario, seed)
          summary_df:  per scenario mean/std/p05/p50/p95 of overall LR + mean stats
          yearly_lr_df: per (scenario, seed, year) loss ratio
    """
    # None -> dynamic entrant count (entrant_base_count * (1 + growth)**year)
    if book_seed is None:
        book_seed = COHORT_CONFIG['seed']
    if overrides is None:
        overrides = [None]
    elif isinstance(overrides, dict):
        overrides = [overrides]
    if isinstance(seeds, int):
        seeds = [seeds]

    tasks = [(i, ov, s) for i, ov in enumerate(overrides) for s in seeds]

    def work(t):
        i, ov, s = t
        cfg = deep_update(COHORT_CONFIG, ov)
        metrics, yearly = _run_one(cfg, s, n_years, new_entrants_per_year, book_seed)
        return i, ov, s, metrics, yearly

    if _HAS_JOBLIB and n_jobs != 1 and len(tasks) > 1:
        out = Parallel(n_jobs=n_jobs, backend='threading')(
            delayed(work)(t) for t in tasks)
    else:
        out = [work(t) for t in tasks]

    rows, yearly_rows = [], []
    for i, ov, s, metrics, yearly in out:
        rows.append({'scenario': i, 'overrides': repr(ov) if ov else 'base',
                     'seed': s, **metrics})
        for year, lr in yearly.items():
            yearly_rows.append({'scenario': i, 'seed': s, 'SIM_YEAR': int(year),
                                'loss_ratio': lr})
    metrics_df = pd.DataFrame(rows)
    yearly_df = pd.DataFrame(yearly_rows)

    summ = []
    for i in range(len(overrides)):
        g = metrics_df[metrics_df['scenario'] == i]
        summ.append({
            'scenario': i,
            'overrides': g['overrides'].iloc[0],
            'n_seeds': len(g),
            'lr_mean': g['overall_lr'].mean(),
            'lr_std': g['overall_lr'].std(),
            'lr_p05': g['overall_lr'].quantile(0.05),
            'lr_p50': g['overall_lr'].median(),
            'lr_p95': g['overall_lr'].quantile(0.95),
            'freq_mean': g['claim_freq'].mean(),
            'premium_mean': g['avg_premium'].mean(),
            'retention_mean': g['retention'].mean(),
        })
    summary_df = pd.DataFrame(summ)
    return metrics_df, summary_df, yearly_df


# Demo: short-horizon illustration (3 years) - not a match to the 20-yr reference run
_demo_metrics, _demo_summary, _demo_yearly = run_monte_carlo(
    overrides = [
        None,
        {'claim_frequency_base': -1.80}
    ],
    seeds = (0,1,2,3,4,5,6,7,8,9,10),
    n_years = 5,
    n_jobs = 5
)

print('\n=== Monte Carlo demo: metrics (scenario x seed) ===')
print(_demo_metrics[['scenario', 'overrides', 'seed', 'overall_lr', 'claim_freq', 'avg_premium', 'retention']].to_string(index=False))
print('\n=== Monte Carlo demo: per-scenario summary')
print(_demo_summary.round(4).to_string(index=False))



=== Monte Carlo demo: metrics (scenario x seed) ===
 scenario                      overrides  seed  overall_lr  claim_freq  avg_premium  retention
        0                           base     0    0.656422    0.140852  1433.224103   0.742038
        0                           base     1    0.691066    0.146108  1432.683522   0.741127
        0                           base     2    0.678342    0.145077  1427.485551   0.739356
        0                           base     3    0.659305    0.145097  1428.428950   0.737022
        0                           base     4    0.690114    0.144463  1434.926995   0.739986
        0                           base     5    0.655600    0.142478  1440.292287   0.739336
        0                           base     6    0.655001    0.143545  1441.343255   0.737170
        0                           base     7    0.664539    0.143919  1447.186992   0.737563
        0                           base     8    0.692950    0.144993  1441.461743   0.7400

In [14]:
# Cohort Analysis Summary

def analyze_cohort(df_cohort):
    """Generate summary statistics from cohort simulation."""
    summary = []
    
    for year in sorted(df_cohort['SIM_YEAR'].unique()):
        year_df = df_cohort[df_cohort['SIM_YEAR'] == year]
        summary.append({
            'Year': year,
            'Active_Policies': len(year_df),
            'Total_Claims': year_df['CLAIM_COUNT'].sum(),
            'Avg_Claims_Per_Policy': year_df['CLAIM_COUNT'].mean(),
            'Total_Claim_Amount': year_df['CLAIM_AMOUNT'].sum(),
            'Avg_Claim_Amount': year_df.loc[
                year_df['CLAIM_COUNT'] > 0, 'CLAIM_AMOUNT'
            ].mean() if year_df['CLAIM_COUNT'].sum() > 0 else 0,
            'Avg_NCD_Level': year_df['NCD_LEVEL'].mean(),
            'Retention_Rate': year_df['RENEWED'].mean()
            if 'RENEWED' in year_df.columns else float('nan'),
            'Avg_Final_Premium': year_df['FINAL_PREMIUM_SST'].mean()
        })
    
    return pd.DataFrame(summary)

# Generate and display cohort summary
cohort_summary = analyze_cohort(cohort_results)
print('=== n-Year Cohort Evolution Summary ===\n')
print(cohort_summary.to_string(index=False))
print(f"\n=== Key Findings ===")
print(f"Total claims over 5 years: {cohort_results['CLAIM_COUNT'].sum():.0f}")
print(f"Total claim cost: RM{cohort_results['CLAIM_AMOUNT'].sum():,.2f}")
print(f"Final avg NCD level: {cohort_results[cohort_results['SIM_YEAR']==2028]['NCD_LEVEL'].mean():.2%}")
print(f"Loss ratio: {cohort_results['CLAIM_AMOUNT'].sum() / cohort_results['FINAL_PREMIUM_SST'].sum():.2%}")

=== n-Year Cohort Evolution Summary ===

 Year  Active_Policies  Total_Claims  Avg_Claims_Per_Policy  Total_Claim_Amount  Avg_Claim_Amount  Avg_NCD_Level  Retention_Rate  Avg_Final_Premium
 2026            10000          1581               0.158100        9.392364e+06       6490.921666       0.314000        0.819900        1553.135703
 2027            13199          2105               0.159482        1.277747e+07       6661.870473       0.340686        0.740511        1458.483325
 2028            14774          2297               0.155476        1.539576e+07       7391.148908       0.357587        0.752200        1416.228102
 2029            16113          2483               0.154099        1.469440e+07       6453.404213       0.365144        0.706572        1398.232622
 2030            16385          2575               0.157156        1.696107e+07       7208.276458       0.368781        0.706683        1385.388288
 2031            16579          2485               0.149888        1.53

In [15]:
# Test: Sample Claim Generation
# Demonstrate claim modeling on sample policies

def generate_sample_claims(df, n=5):
    """Generate sample claims for testing the model."""
    samples = df.sample(n=min(n, len(df)), random_state=42)

    print('=== Sample Claim Generation ===\n')

    for idx, row in samples.iterrows():
        lamb = row['CLAIM_LAMBDA']
        n_claims = np.random.poisson(lamb)

        print(f"Policy: {row['POLID'][:12]}...")
        print(f"  Coverage: {row['COVERAGE_TYPE']}, Vehicle: {row['VEHICLE_TYPE']}")
        print(f"  Sum Assured: RM{row['SUM_ASSURED']:,.0f}")
        print(f"  Flood Risk: {row['FLOOD_RISK']}, Theft Risk: {row['THEFT_RISK']}")
        print(f"  NCD Years: {row['NCD_YEARS']} ({row['NCD_LEVEL']:.1%})")
        print(f"  Claim Intensity (lambda): {lamb:.3f}")

        if n_claims > 0:
            total_claim, perils = generate_claim_total(
                row['COVERAGE_TYPE'], row['SUM_ASSURED'], n_claims
            )
            print(f"  Perils: {perils}")
            print(f"  TOTAL CLAIM: RM{total_claim:,.2f}")
        else:
            print('  No claims this year')

        print(f"  Basic Premium: RM{row['BASIC_PREMIUM']:,.2f}")
        print(f"  Final Premium (w/ SST): RM{row['FINAL_PREMIUM_SST']:,.2f}")
        print()


# Run sample generation
generate_sample_claims(df, n=1)


=== Sample Claim Generation ===

Policy: INIT2026-006...
  Coverage: Comprehensive, Vehicle: ICE
  Sum Assured: RM23,000
  Flood Risk: False, Theft Risk: False
  NCD Years: 0 (0.0%)
  Claim Intensity (lambda): 0.153
  No claims this year
  Basic Premium: RM1,041.60
  Final Premium (w/ SST): RM1,322.92



In [16]:

# ============================================================
# Statistical Validation (Part A: Core Tests)
# ============================================================

results = cohort_results.copy()
passed = []
failed = []

def check(name, cond, detail=''):
    if cond:
        passed.append(name)
        print(f"[PASS] {name}")
    else:
        failed.append(name)
        print(f"[FAIL] {name} {detail}")

# Test 1: Overall claim frequency in plausible band (10%-20%)
overall_freq = results['CLAIM_OCCURRED'].mean()
check('1. Overall claim frequency within 10%-20%',
      0.10 <= overall_freq <= 0.20,
      f"(actual {overall_freq:.1%})")

# Test 2: TPO expected claim cost per policy-year < Comprehensive
# (frequency x mean severity - TPO has no own-damage exposure)
comp_freq_t = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq_t = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
comp_sev = results.loc[(results['COVERAGE_TYPE']=='Comprehensive') &
                       (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
tpo_sev = results.loc[(results['COVERAGE_TYPE']=='TPO') &
                      (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
comp_mean = comp_sev.mean() if len(comp_sev) else 0
tpo_mean = tpo_sev.mean() if len(tpo_sev) else 0
comp_cost = comp_freq_t * comp_mean
tpo_cost = tpo_freq_t * tpo_mean
check('2. TPO expected claim cost/policy-year < Comprehensive',
      tpo_cost < comp_cost,
      f"(Comp RM{comp_cost:,.0f} vs TPO RM{tpo_cost:,.0f})")

# Test 3: Per-peril means (report; caps enforced at draw time)
peril_means = (results[results['CLAIM_AMOUNT']>0]
               .assign(peril_first=lambda d: d['CLAIM_PERIL'].str.split('/').str[0])
               .groupby('peril_first')['CLAIM_AMOUNT'].mean())
print('\n  Per-peril mean severity:')
for peril, m in peril_means.sort_values(ascending=False).items():
    print(f"    {peril:10s} RM{m:,.0f}  (n={len(results[results['CLAIM_PERIL'].str.contains(peril)])})")
print('  (caps enforced at draw time by construction)')

# Test 4: Loss ratio (incurred / earned premium) - report only
earned = results['FINAL_PREMIUM_SST'].sum()
incurred = results['CLAIM_AMOUNT'].sum()
loss_ratio = incurred / earned if earned > 0 else float('nan')
print(f"\n  Loss ratio: {loss_ratio:.1%} (earned RM{earned:,.0f}, incurred RM{incurred:,.0f})")
if not (0.40 <= loss_ratio <= 0.80):
    print(f"  [WARN] Loss ratio outside 40%-80% band - inspect premium adequacy")
else:
    print(f"  [OK]   Loss ratio within 40%-80% band")

# Test 5: NCD mechanics - claim resets to 0, claim-free increments
reset_ok = (results.loc[results['CLAIM_OCCURRED'], 'NCD_YEARS'] == 0).mean()
inc_ok = (results.loc[~results['CLAIM_OCCURRED'], 'NCD_YEARS'] >= 1).mean()
check('5a. Claims reset NCD_YEARS to 0', reset_ok >= 0.99,
      f"(reset rate {reset_ok:.1%})")
check('5b. Claim-free years increment NCD', inc_ok >= 0.99,
      f"(increment rate {inc_ok:.1%})")

# Test 6: TPO frequency < Comprehensive frequency
comp_freq = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
check('6. TPO claim frequency < Comprehensive',
      tpo_freq < comp_freq,
      f"(Comp {comp_freq:.1%} vs TPO {tpo_freq:.1%})")

# Test 7: Data integrity - no nulls/negatives in key columns
key_cols = ['CLAIM_COUNT', 'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'FINAL_PREMIUM_SST',
            'NCD_LEVEL', 'NCD_YEARS', 'RENEWED', 'CAR_AGE', 'TOTAL_LOADING',
            'NCD_LEVEL_PRICED']
null_bad = results[key_cols].isnull().sum().sum()
neg_bad = (results['CLAIM_AMOUNT'] < 0).sum() + (results['FINAL_PREMIUM_SST'] <= 0).sum()
check('7. No nulls in key columns', null_bad == 0, f"(nulls: {null_bad})")
check('7b. No negative/zero premium or negative claims', neg_bad == 0,
      f"(bad: {neg_bad})")

# Test 8b: Ageing works - in-force DRIVER_AGE/CAR_AGE increase across years
age_by_year = results.groupby('SIM_YEAR')[['DRIVER_AGE', 'CAR_AGE']].mean()
age_growth = age_by_year['DRIVER_AGE'].iloc[-1] > age_by_year['DRIVER_AGE'].iloc[0]
car_growth = age_by_year['CAR_AGE'].iloc[-1] > age_by_year['CAR_AGE'].iloc[0]
check('8b. Mean DRIVER_AGE rises across years', age_growth,
      f"({age_by_year['DRIVER_AGE'].iloc[0]:.1f} -> {age_by_year['DRIVER_AGE'].iloc[-1]:.1f})")
check('8c. Mean CAR_AGE rises across years', car_growth,
      f"({age_by_year['CAR_AGE'].iloc[0]:.1f} -> {age_by_year['CAR_AGE'].iloc[-1]:.1f})")

# Test 9: CAR_AGE plausibility
# Fleet age at inception (year-1 policies), not the aged 20-yr book
car_age_med = results.loc[results['SIM_YEAR'] == COHORT_CONFIG['cohort_year'], 'CAR_AGE'].median()
check('9. CAR_AGE median within 3-4 years', 3.0 <= car_age_med <= 4.0,
      f"(median {car_age_med:.1f})")
check('9b. CAR_AGE within [0,10]', results['CAR_AGE'].between(0, 10).all())
genz_car = results.loc[results['DRIVER_AGE_CAT'] == 'Young Adults', 'CAR_AGE'].mean()
other_car = results.loc[results['DRIVER_AGE_CAT'] != 'Young Adults', 'CAR_AGE'].mean()
check('9c. Young Adults drive newer cars than other bands', genz_car < other_car,
      f"(Young Adults {genz_car:.1f} vs others {other_car:.1f})")

# Test 10: Premium evolves across years for in-force policies
years_per_polid = results.groupby('POLID')['SIM_YEAR'].nunique()
multi_year = results[results['POLID'].isin(
    years_per_polid[years_per_polid >= 3].index
)]
prem_levels = multi_year.groupby('POLID')['FINAL_PREMIUM_SST'].nunique()
prem_varies = (prem_levels > 1).mean()
check('10. Premium varies across years for multi-year policies',
      prem_varies >= 0.9, f"({prem_varies:.1%} of policies vary)")

# Test 11: Dynamic entrants - EV adoption ramp + growth-based count
_sched = COHORT_CONFIG.get('ev_share_by_year') or {}
_has_ramp = len(_sched) > 1 and (max(_sched.values()) - min(_sched.values())) > 1e-9
ent = results[results['POLID'].str.startswith('ENT')]
if not _has_ramp:
    print('[SKIP] Test 11 (EV share constant or unset)')
elif len(ent) and len(ent['SIM_YEAR'].unique()) > 1:
    ev_by_year = ent.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
    check('11a. Entrant EV share rises across years',
          ev_by_year.iloc[-1] > ev_by_year.iloc[0],
          f"({ev_by_year.iloc[0]:.1%} -> {ev_by_year.iloc[-1]:.1%})")
    check('11b. Entrant EV share <= 0.95', ev_by_year.max() <= 0.95,
          f"(max {ev_by_year.max():.1%})")
    all_ev = results.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
    check('11c. Portfolio EV share rises across years',
          all_ev.iloc[-1] > all_ev.iloc[0],
          f"({all_ev.iloc[0]:.1%} -> {all_ev.iloc[-1]:.1%})")
    if COHORT_CONFIG['entrant_annual_growth'] > 0:
        ent_cnt = ent.groupby('SIM_YEAR')['POLID'].count()
        check('11d. Entrant count grows with annual growth',
              ent_cnt.iloc[-1] > ent_cnt.iloc[0],
              f"({ent_cnt.iloc[0]:,} -> {ent_cnt.iloc[-1]:,})")
    else:
        print('[SKIP] 11d. Entrant count growth (annual growth = 0)')
else:
    print('[SKIP] Test 11 (no multi-year entrant data)')

print(f"\n===== VALIDATION RESULT: {len(passed)} passed, {len(failed)} failed =====")


[PASS] 1. Overall claim frequency within 10%-20%
[PASS] 2. TPO expected claim cost/policy-year < Comprehensive

  Per-peril mean severity:
    TPBI       RM26,293  (n=2930)
    Theft      RM13,422  (n=5241)
    Fire       RM8,937  (n=2530)
    TPPD       RM5,489  (n=11005)
    AD         RM4,285  (n=20773)
    Windscreen RM1,906  (n=5533)
  (caps enforced at draw time by construction)

  Loss ratio: 69.0% (earned RM451,868,858, incurred RM311,820,123)
  [OK]   Loss ratio within 40%-80% band
[PASS] 5a. Claims reset NCD_YEARS to 0
[PASS] 5b. Claim-free years increment NCD
[PASS] 6. TPO claim frequency < Comprehensive
[PASS] 7. No nulls in key columns
[PASS] 7b. No negative/zero premium or negative claims
[PASS] 8b. Mean DRIVER_AGE rises across years
[PASS] 8c. Mean CAR_AGE rises across years
[PASS] 9. CAR_AGE median within 3-4 years
[PASS] 9b. CAR_AGE within [0,10]
[PASS] 9c. Young Adults drive newer cars than other bands
[PASS] 10. Premium varies across years for multi-year policies
[SK

In [17]:

# ============================================================
# Statistical Validation (Part B) + EDA Enrichment
# ============================================================

# Test 8: Reproducibility - re-running with same seed gives identical claims
np.random.seed(999)
run_a = simulate_cohort(df, n_years=3, seed=7)

np.random.seed(999)
run_b = simulate_cohort(df, n_years=3, seed=7)

same_claims = (run_a['CLAIM_COUNT'].values == run_b['CLAIM_COUNT'].values).all()
same_amounts = np.allclose(run_a['CLAIM_AMOUNT'].values, run_b['CLAIM_AMOUNT'].values)

if same_claims and same_amounts:
    print("[PASS] 8. Reproducibility: same seed -> identical claims")
else:
    print("[FAIL] 8. Reproducibility: seeded runs differ")

# ---- EDA enrichment ----
res = cohort_results.copy()

# E1: Loss ratio by coverage type (n/a if no earned premium)
def lr_ratio(d):
    earned = d['FINAL_PREMIUM_SST'].sum()
    if earned <= 0:
        return 'n/a (no earned premium)'
    return f"{d['CLAIM_AMOUNT'].sum() / earned:.1%}"
lr_by_cov = res.groupby('COVERAGE_TYPE').apply(lr_ratio)
print('\nE1. Loss ratio by coverage type (TPO underpriced - see FINDING 1):')
print(lr_by_cov.to_string())

# E2: Peril mix across all claims
peril_counts = res[res['CLAIM_AMOUNT'] > 0]['CLAIM_PERIL'].str.split('/').explode().value_counts()
print('\nE2. Peril mix:')
print(peril_counts.to_string())

# E3: Claim frequency by age category
freq_by_age = res.groupby('DRIVER_AGE_CAT')['CLAIM_OCCURRED'].mean().sort_values(ascending=False)
print('\nE3. Claim frequency by age category:')
print(freq_by_age.map(lambda x: f"{x:.1%}").to_string())

# E4: Retention by claim status
ret_by_claim = res.groupby('CLAIM_OCCURRED')['RENEWED'].mean()
print('\nE4. Retention by claim status:')
print(ret_by_claim.map(lambda x: f"{x:.1%}").to_string())


Year 2026: 10000 active policies, claims: 1668, freq: 15.4%, avg NCD priced: 24.79%, avg premium: RM1553.14, retention: 81.5%
Year 2027: 13147 active policies, claims: 2127, freq: 14.7%, avg NCD priced: 29.77%, avg premium: RM1474.29, retention: 74.3%
Year 2028: 14774 active policies, claims: 2304, freq: 14.3%, avg NCD priced: 32.28%, avg premium: RM1442.03, retention: 75.2%

Simulation complete. Total records: 37921
Year range: 2026 - 2028
Simulation wall time: 0.5s
Year 2026: 10000 active policies, claims: 1668, freq: 15.4%, avg NCD priced: 24.79%, avg premium: RM1553.14, retention: 81.5%
Year 2027: 13147 active policies, claims: 2127, freq: 14.7%, avg NCD priced: 29.77%, avg premium: RM1474.29, retention: 74.3%
Year 2028: 14774 active policies, claims: 2304, freq: 14.3%, avg NCD priced: 32.28%, avg premium: RM1442.03, retention: 75.2%

Simulation complete. Total records: 37921
Year range: 2026 - 2028
Simulation wall time: 0.5s
[PASS] 8. Reproducibility: same seed -> identical claims

In [18]:

# ============================================================
# Individual Policy Trajectories (premium evolution over years)
# ============================================================

def plot_policy_trajectories(df_cohort, n_policies=20, min_years=3,
                             out='images/policy-trajectories.png', seed=42):
    """Plot premium + priced-NCD trajectories for random multi-year policies."""
    years_per_polid = df_cohort.groupby('POLID')['SIM_YEAR'].nunique()
    eligible = years_per_polid[years_per_polid >= min_years].index.tolist()
    rng = np.random.RandomState(seed)
    picks = [str(p) for p in rng.choice(
        eligible, size=min(n_policies, len(eligible)), replace=False
    )]

    sel = df_cohort[df_cohort['POLID'].isin(picks)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for pid in picks:
        p = sel[sel['POLID'] == pid].sort_values('SIM_YEAR')
        axes[0].plot(p['SIM_YEAR'], p['FINAL_PREMIUM_SST'], marker='o',
                     label=f"{pid[:12]}...")
        axes[1].plot(p['SIM_YEAR'], p['NCD_LEVEL_PRICED'], marker='s')
    axes[0].set_title('Final premium by year')
    axes[0].set_xlabel('SIM_YEAR')
    axes[0].set_ylabel('FINAL_PREMIUM_SST (RM)')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[1].set_title('NCD used for pricing by year')
    axes[1].set_xlabel('SIM_YEAR')
    axes[1].set_ylabel('NCD_LEVEL_PRICED')
    axes[1].grid(alpha=0.3)
    fig.tight_layout()

    os.makedirs(os.path.dirname(out), exist_ok=True)
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Trajectory plot saved: {out}")

    print('\n=== Selected policy trajectories (year-by-year) ===')
    cols = ['POLID', 'SIM_YEAR', 'COVERAGE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
            'CAR_AGE', 'NCD_LEVEL_PRICED', 'FINAL_PREMIUM_SST', 'CLAIM_OCCURRED']
    print(sel.sort_values(['POLID', 'SIM_YEAR'])[cols].to_string(index=False))


plot_policy_trajectories(cohort_results)


Trajectory plot saved: images/policy-trajectories.png

=== Selected policy trajectories (year-by-year) ===
          POLID  SIM_YEAR COVERAGE_TYPE DRIVER_AGE_CAT  DRIVER_AGE  CAR_AGE  NCD_LEVEL_PRICED  FINAL_PREMIUM_SST  CLAIM_OCCURRED
 ENT2027-001645      2027 Comprehensive   Young Adults          23        4            0.0000            3713.95           False
 ENT2027-001645      2028 Comprehensive   Young Adults          24        5            0.2500            2860.08           False
 ENT2027-001645      2029 Comprehensive   Young Adults          25        6            0.3000            2739.04           False
 ENT2027-001645      2030 Comprehensive   Young Adults          26        7            0.3833            2474.45           False
 ENT2028-001703      2028 Comprehensive  Mature Adults          63        6            0.2500            7022.72           False
 ENT2028-001703      2029 Comprehensive  Mature Adults          64        7            0.3000            6721.18       

In [19]:
# ============================================================================
# OVERTHINKER-STYLE VALIDATION (Part A + Enhanced Tests + Correlation)
# Adapted from reference overthinker_gen_data.ipynb Section 6
# ============================================================================

final_dataset = cohort_results.copy()

print("="*70)
print("DATASET VALIDATION")
print("="*70)

validation_results = []

# 1. Premium vs Sum Insured Correlation (Comprehensive; TPO premium is SA-free by design)
comp_sub = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']
corr_comp = comp_sub['FINAL_PREMIUM_SST'].corr(comp_sub['SUM_ASSURED'])
validation_results.append({
    'Test': 'Premium vs Sum Insured Correlation (Comp)',
    'Value': f"{corr_comp:.3f}",
    'Expected': '> 0.5',
    'Pass': corr_comp > 0.5
})

# 2. Young Driver Premium Loading
young_avg = final_dataset[final_dataset['DRIVER_AGE'] < 25]['FINAL_PREMIUM_SST'].mean()
mature_avg = final_dataset[final_dataset['DRIVER_AGE'].between(30, 50)]['FINAL_PREMIUM_SST'].mean()
loading = young_avg / mature_avg
validation_results.append({
    'Test': 'Young Driver Premium Loading',
    'Value': f"{loading:.2f}x",
    'Expected': '> 1.05x (driver loading partly offset by newer cars)',
    'Pass': loading > 1.05
})

# 3. Claim Frequency Range
claim_freq = (final_dataset['CLAIM_COUNT'] > 0).mean()
validation_results.append({
    'Test': 'Overall Claim Frequency',
    'Value': f"{claim_freq*100:.2f}%",
    'Expected': '10-20%',
    'Pass': 0.10 <= claim_freq <= 0.20
})

# 4. NCD Progression
ncd_2026 = final_dataset[final_dataset['SIM_YEAR'] == 2026]['NCD_LEVEL_PRICED'].mean()
ncd_2030 = final_dataset[final_dataset['SIM_YEAR'] == 2030]['NCD_LEVEL_PRICED'].mean()
validation_results.append({
    'Test': 'NCD Progression (2026 < 2030)',
    'Value': f"{ncd_2026*100:.1f}% to {ncd_2030*100:.1f}%",
    'Expected': 'Increasing',
    'Pass': ncd_2030 > ncd_2026
})

# 5. Loss Ratio Range
loss_ratio = final_dataset['CLAIM_AMOUNT'].sum() / final_dataset['FINAL_PREMIUM_SST'].sum()
validation_results.append({
    'Test': 'Overall Loss Ratio',
    'Value': f"{loss_ratio:.2%}",
    'Expected': '50-80%',
    'Pass': 0.50 <= loss_ratio <= 0.80
})

# 6. Region premium difference (report only - rate file dependent)
east_avg = final_dataset[final_dataset['REGION'].str.contains('East')]['FINAL_PREMIUM_SST'].mean()
pen_avg = final_dataset[final_dataset['REGION'].str.contains('Peninsular')]['FINAL_PREMIUM_SST'].mean()
region_ratio = east_avg / pen_avg if pen_avg > 0 else float('nan')
print(f"  Region premium ratio (East/Peninsular): {region_ratio:.2f}x (report only)")

# 7. Comprehensive vs TPO Premium
comp_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']['FINAL_PREMIUM_SST'].mean()
tpo_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPO']['FINAL_PREMIUM_SST'].mean()
comp_ratio = comp_avg / tpo_avg if tpo_avg > 0 else float('nan')
validation_results.append({
    'Test': 'Comprehensive Premium > TPO',
    'Value': f"{comp_ratio:.2f}x",
    'Expected': '> 1.8x',
    'Pass': comp_ratio > 1.8
})

# 7b. TPFT premium between TPO and Comprehensive
tpft_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPFT']['FINAL_PREMIUM_SST'].mean()
validation_results.append({
    'Test': 'TPFT Premium between TPO & Comp',
    'Value': f"RM{tpo_avg:,.0f} < RM{tpft_avg:,.0f} < RM{comp_avg:,.0f}",
    'Expected': 'TPO < TPFT < Comprehensive',
    'Pass': tpo_avg < tpft_avg < comp_avg
})

# 8. NCD Discount (controlled: same policy priced at NCD=0 vs actual)
def premium_at_zero_ncd(row):
    r = row.copy()
    r['NCD_LEVEL'] = 0.0
    return compute_final_premium(r)

sample = final_dataset.sample(min(30000, len(final_dataset)), random_state=42).copy()
sample['PREMIUM_NCD0'] = sample.apply(premium_at_zero_ncd, axis=1)
sample['NCD_DISCOUNT'] = 1 - sample['FINAL_PREMIUM_SST'] / sample['PREMIUM_NCD0']
max_ncd_disc = sample.loc[sample['NCD_LEVEL_PRICED'] == 0.55, 'NCD_DISCOUNT'].median()
validation_results.append({
    'Test': 'NCD Discount at max tier (controlled)',
    'Value': f"{max_ncd_disc*100:.1f}%",
    'Expected': '45-60%',
    'Pass': 0.45 <= max_ncd_disc <= 0.60
})

# 9. Comprehensive Premium Trend (2026 -> 2030)
comp_2026_avg = final_dataset[(final_dataset['SIM_YEAR'] == 2026) & (final_dataset['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
comp_2030_avg = final_dataset[(final_dataset['SIM_YEAR'] == 2030) & (final_dataset['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
comp_trend = comp_2030_avg / comp_2026_avg if comp_2026_avg > 0 else float('nan')
validation_results.append({
    'Test': 'Comp Premium Trend (2026 to 2030)',
    'Value': f"RM{comp_2026_avg:,.0f} to RM{comp_2030_avg:,.0f} ({comp_trend:.2f}x)",
    'Expected': 'Mild softening (0.80x-0.99x)',
    'Pass': 0.80 <= comp_trend < 1.00
})

validation_df = pd.DataFrame(validation_results)
print("\n" + validation_df.to_string(index=False))
all_pass = validation_df['Pass'].all()
print("\n" + "="*70)
if all_pass:
    print("ALL VALIDATION TESTS PASSED")
else:
    print("SOME VALIDATION TESTS FAILED - REVIEW ABOVE")
print("="*70)

# ---- Correlation Heatmap ----
numeric_cols = ['DRIVER_AGE', 'CAR_AGE', 'SUM_ASSURED', 'FINAL_PREMIUM_SST',
                'TOTAL_LOADING', 'NCD_LEVEL_PRICED', 'CLAIM_COUNT',
                'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'NCD_YEARS']
correlation_matrix = final_dataset[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('Correlation Matrix - Key Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nCorrelation heatmap saved: images/correlation_heatmap.png")

# ============================================================================
# PART B: ENHANCED STATISTICAL TESTS
# ============================================================================
print("\n" + "="*70)
print("ENHANCED STATISTICAL VALIDATION")
print("="*70)

enhanced_results = []

# 1. Premium Log-Normal Fit (KS)
log_premiums = np.log(final_dataset['FINAL_PREMIUM_SST'])
ks_stat, ks_pval = kstest(log_premiums, norm(loc=log_premiums.mean(), scale=log_premiums.std()).cdf)
enhanced_results.append({
    'Test': 'Premium Log-Normal Fit (KS test)',
    'Statistic': f"KS={ks_stat:.4f}",
    'P-value': f"{ks_pval:.4e}",
    'Interpretation': 'Approx log-normal' if ks_stat < 0.1 else 'Mixture (TPO+Comp+NCD)',
    'Pass': ks_stat < 0.20
})
print("\n1. PREMIUM DISTRIBUTION FIT TEST (Log-Normal)")
print(f"   KS Statistic: {ks_stat:.4f}, p-value: {ks_pval:.4e}")
print(f"   (N={len(final_dataset):,}; threshold 0.15 given coverage/NCD mixture)")

# 2. Severity Gamma Fit by coverage
print("\n2. CLAIM SEVERITY DISTRIBUTION FIT (Gamma)")
print("-"*60)
for cov_type in ['Comprehensive', 'TPFT', 'TPO']:
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov_type) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 30:
        shape_fit, loc_fit, scale_fit = gamma_dist.fit(claims_subset, floc=0)
        ks_s, ks_p = kstest(claims_subset, gamma_dist(a=shape_fit, loc=loc_fit, scale=scale_fit).cdf)
        enhanced_results.append({
            'Test': f'Severity Gamma Fit ({cov_type})',
            'Statistic': f"shape={shape_fit:.2f}, KS={ks_s:.4f}",
            'P-value': f"{ks_p:.4e}",
            'Interpretation': f'Per-peril mixture, shape={shape_fit:.2f}',
            'Pass': ks_s < 0.20
        })
        print(f"   {cov_type}: shape={shape_fit:.2f}, scale={scale_fit:.0f}, KS={ks_s:.4f}")
    else:
        print(f"   {cov_type}: Insufficient claims (n={len(claims_subset)})")

# 3. NCD Distribution by Year
print("\n3. NCD DISTRIBUTION BY YEAR")
print("-"*60)
ncd_by_year = final_dataset.groupby('SIM_YEAR')['NCD_LEVEL_PRICED'].value_counts(normalize=True).unstack(fill_value=0)
ncd_by_year = ncd_by_year.reindex(columns=sorted(ncd_by_year.columns))
for year in sorted(final_dataset['SIM_YEAR'].unique()):
    yd = final_dataset[final_dataset['SIM_YEAR'] == year]
    print(f"   {year}: Avg NCD={yd['NCD_LEVEL_PRICED'].mean()*100:.1f}% | NCD=0%: {(yd['NCD_LEVEL_PRICED']==0.0).mean()*100:.1f}% | NCD=55%: {(yd['NCD_LEVEL_PRICED']==0.55).mean()*100:.1f}%")

ncd_2026_max = (final_dataset[final_dataset['SIM_YEAR']==2026]['NCD_LEVEL_PRICED']==0.55).mean()
ncd_2030_max = (final_dataset[final_dataset['SIM_YEAR']==2030]['NCD_LEVEL_PRICED']==0.55).mean()
enhanced_results.append({
    'Test': 'NCD: 55% tier grows over years',
    'Statistic': f"{ncd_2026_max*100:.1f}% to {ncd_2030_max*100:.1f}%",
    'P-value': '-',
    'Interpretation': 'Loyal claim-free policies accumulate',
    'Pass': ncd_2030_max > ncd_2026_max
})

# NCD distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ncd_labels = sorted(final_dataset['NCD_LEVEL_PRICED'].unique())
ncd_year_data = []
for year in sorted(final_dataset['SIM_YEAR'].unique()):
    year_ncd = final_dataset[final_dataset['SIM_YEAR'] == year]['NCD_LEVEL_PRICED']
    row = {f'{n*100:.0f}%': (year_ncd == n).mean()*100 for n in ncd_labels}
    row['Year'] = year
    ncd_year_data.append(row)
ncd_plot_df = pd.DataFrame(ncd_year_data).set_index('Year')
ncd_plot_df.plot(kind='bar', stacked=True, ax=axes[0], colormap='YlOrRd_r')
axes[0].set_title('NCD Distribution by Year', fontsize=14, fontweight='bold')
axes[0].set_ylabel('% of Policies')
axes[0].legend(title='NCD %', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[0].set_xlabel('SIM_YEAR')

tier_labels = {0.0: '0%', 0.25: '25%', 0.30: '30%', 0.3833: '38%', 0.45: '45%', 0.55: '55%'}
final_dataset['NCD_TIER'] = final_dataset['NCD_LEVEL_PRICED'].map(tier_labels)
final_dataset.boxplot(column='FINAL_PREMIUM_SST', by='NCD_TIER', ax=axes[1])
axes[1].set_title('Premium Distribution by NCD Tier', fontsize=14, fontweight='bold')
axes[1].set_xlabel('NCD Tier')
axes[1].set_ylabel('Premium (RM)')
axes[1].get_figure().suptitle('')
plt.tight_layout()
plt.savefig('images/ncd_validation.png', dpi=150, bbox_inches='tight')
plt.close()

# 4. Loss Ratio by Segment
print("\n4. LOSS RATIO BY SEGMENT")
print("-"*60)
total_premium = final_dataset['FINAL_PREMIUM_SST'].sum()
total_incurred = final_dataset['CLAIM_AMOUNT'].sum()
overall_lr = total_incurred / total_premium
print("   By Coverage Type:")
for cov in ['Comprehensive', 'TPFT', 'TPO']:
    subset = final_dataset[final_dataset['COVERAGE_TYPE'] == cov]
    lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
    print(f"     {cov:14s}: LR={lr:.2%} (n={len(subset):,})")
final_dataset['age_band'] = pd.cut(final_dataset['DRIVER_AGE'],
                                   bins=[17, 25, 35, 50, 65, 100],
                                   labels=['18-25', '26-35', '36-50', '51-65', '66+'])
print("   By Age Band:")
for band in ['18-25', '26-35', '36-50', '51-65', '66+']:
    subset = final_dataset[final_dataset['age_band'] == band]
    if len(subset) > 0 and subset['FINAL_PREMIUM_SST'].sum() > 0:
        lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
        print(f"     {band:6s}: LR={lr:.2%} (n={len(subset):,})")
print("   By Region:")
for loc in ['Peninsular Malaysia', 'East Malaysia (Sabah, Sawarak & Labuan)']:
    subset = final_dataset[final_dataset['REGION'] == loc]
    lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
    print(f"     {loc[:12]:12s}: LR={lr:.2%}")
enhanced_results.append({
    'Test': 'Loss Ratio in Actuarial Range',
    'Statistic': f"LR={overall_lr:.2%}",
    'P-value': '-',
    'Interpretation': 'Typically 50-80% for motor',
    'Pass': 0.40 <= overall_lr <= 0.90
})

# 5. Severity Percentile Validation
print("\n5. SEVERITY PERCENTILE VALIDATION")
print("-"*60)
severity_targets = {
    'Comprehensive': {'mean': (4000, 10000), 'p95': (15000, 60000)},
    'TPO': {'mean': (8000, 30000), 'p95': (25000, 600000)},
    'TPFT': {'mean': (5000, 15000), 'p95': (20000, 120000)}
}
for cov_type, targets in severity_targets.items():
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov_type) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 10:
        mean_sev = claims_subset.mean()
        p95_sev = claims_subset.quantile(0.95)
        mean_pass = targets['mean'][0] <= mean_sev <= targets['mean'][1]
        p95_pass = targets['p95'][0] <= p95_sev <= targets['p95'][1]
        print(f"   {cov_type}: Mean=RM{mean_sev:,.0f} (target RM{targets['mean'][0]:,}-{targets['mean'][1]:,}) {'OK' if mean_pass else 'CHECK'}")
        print(f"     P95=RM{p95_sev:,.0f} (target RM{targets['p95'][0]:,}-{targets['p95'][1]:,}) {'OK' if p95_pass else 'CHECK'}")
        enhanced_results.append({'Test': f'Severity Mean ({cov_type})', 'Statistic': f"RM{mean_sev:,.0f}", 'P-value': '-', 'Interpretation': f'Target RM{targets["mean"][0]:,}-{targets["mean"][1]:,}', 'Pass': mean_pass})
        enhanced_results.append({'Test': f'Severity P95 ({cov_type})', 'Statistic': f"RM{p95_sev:,.0f}", 'P-value': '-', 'Interpretation': f'Target RM{targets["p95"][0]:,}-{targets["p95"][1]:,}', 'Pass': p95_pass})

# 6. Longitudinal Consistency (POLID tracking)
print("\n6. LONGITUDINAL CONSISTENCY")
print("-"*60)
ph_counts = final_dataset.groupby('POLID')['SIM_YEAR'].nunique()
print(f"   Unique policies: {len(ph_counts):,}")
print(f"   Policies with 1 year: {(ph_counts == 1).sum():,}")
print(f"   Policies with 2+ years: {(ph_counts >= 2).sum():,}")
print(f"   Policies with all 5 years: {(ph_counts == 5).sum():,}")
multi_year_phs = ph_counts[ph_counts >= 2].index[:1000]
age_errors = 0
car_errors = 0
ncd_reset_failures = 0
total_claims_checked = 0
for ph_id in multi_year_phs:
    ph = final_dataset[final_dataset['POLID'] == ph_id].sort_values('SIM_YEAR')
    for i in range(1, len(ph)):
        year_diff = ph.iloc[i]['SIM_YEAR'] - ph.iloc[i-1]['SIM_YEAR']
        age_diff = ph.iloc[i]['DRIVER_AGE'] - ph.iloc[i-1]['DRIVER_AGE']
        car_diff = ph.iloc[i]['CAR_AGE'] - ph.iloc[i-1]['CAR_AGE']
        if age_diff != year_diff:
            age_errors += 1
        if car_diff != year_diff and not (car_diff == 0 and ph.iloc[i]['CAR_AGE'] == 10):
            car_errors += 1
        prev_claim = ph.iloc[i-1]['CLAIM_OCCURRED']
        curr_ncd = ph.iloc[i]['NCD_LEVEL_PRICED']
        if prev_claim:
            total_claims_checked += 1
            if curr_ncd > 0:
                ncd_reset_failures += 1
age_ok = age_errors == 0
car_ok = car_errors == 0
ncd_ok = ncd_reset_failures == 0
print(f"\n   Age consistency (sample {len(multi_year_phs):,}): errors={age_errors} {'OK' if age_ok else 'CHECK'}")
print(f"   Car-age consistency (cap at 10 allowed): errors={car_errors} {'OK' if car_ok else 'CHECK'}")
print(f"   NCD reset on claim: failures={ncd_reset_failures}/{total_claims_checked} {'OK' if ncd_ok else 'CHECK'}")
enhanced_results.append({'Test': 'Longitudinal: Age Consistency', 'Statistic': f"{age_errors} errors", 'P-value': '-', 'Interpretation': 'DRIVER_AGE +1 per year', 'Pass': age_ok})
enhanced_results.append({'Test': 'Longitudinal: Car-Age Consistency', 'Statistic': f"{car_errors} errors", 'P-value': '-', 'Interpretation': 'CAR_AGE +1/yr (cap 10)', 'Pass': car_ok})
enhanced_results.append({'Test': 'NCD Reset on Claim', 'Statistic': f"{ncd_reset_failures}/{total_claims_checked} failures", 'P-value': '-', 'Interpretation': 'Claim in year N -> NCD 0 next year', 'Pass': ncd_ok})

# 7b. TPFT claim frequency between TPO and Comprehensive
comp_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']['CLAIM_OCCURRED'].mean()
tpo_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPO']['CLAIM_OCCURRED'].mean()
tpft_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPFT']['CLAIM_OCCURRED'].mean()
enhanced_results.append({'Test': 'TPFT Claim Frequency between TPO & Comp', 'Statistic': f"{tpo_freq_c:.1%} < {tpft_freq_c:.1%} < {comp_freq_c:.1%}", 'P-value': '-', 'Interpretation': 'TPFT covers TP + fire/theft only', 'Pass': tpo_freq_c < tpft_freq_c < comp_freq_c})

# 7. Young Adults vs older drivers
print("\n7. GEN Z vs NON-GEN Z COMPARISON")
print("-"*60)
gen_z = final_dataset[final_dataset['DRIVER_AGE'] <= 27]
non_gen_z = final_dataset[final_dataset['DRIVER_AGE'] > 27]
gen_z_pct = len(gen_z) / len(final_dataset) * 100
gz_claim = (gen_z['CLAIM_COUNT'] > 0).mean()
nz_claim = (non_gen_z['CLAIM_COUNT'] > 0).mean()
gz_prem = gen_z['FINAL_PREMIUM_SST'].mean()
nz_prem = non_gen_z['FINAL_PREMIUM_SST'].mean()
print(f"   Young Adults share: {gen_z_pct:.1f}%")
print(f"   Claim rate: Young Adults={gz_claim*100:.1f}% vs Older={nz_claim*100:.1f}%")
print(f"   Avg premium: Young Adults=RM{gz_prem:,.0f} vs Older=RM{nz_prem:,.0f}")
enhanced_results.append({'Test': 'Young Adults Share', 'Statistic': f"{gen_z_pct:.1f}%", 'P-value': '-', 'Interpretation': 'Target 25-40%', 'Pass': 25 <= gen_z_pct <= 40})
enhanced_results.append({'Test': 'Young Adults Higher Claim Rate', 'Statistic': f"{gz_claim*100:.1f}% vs {nz_claim*100:.1f}%", 'P-value': '-', 'Interpretation': 'Young drivers claim more', 'Pass': gz_claim > nz_claim})

# 8. Claim Count Poisson Fit (Var/Mean)
print("\n8. CLAIM COUNT DISTRIBUTION")
print("-"*60)
claim_counts = final_dataset['CLAIM_COUNT']
mean_claims = claim_counts.mean()
var_mean = claim_counts.var() / mean_claims
print(f"   Mean: {mean_claims:.4f}, Var/Mean: {var_mean:.3f} (1.0 = perfect Poisson)")
enhanced_results.append({'Test': 'Claim Count Poisson Fit (Var/Mean)', 'Statistic': f"{var_mean:.3f}", 'P-value': '-', 'Interpretation': 'Close to 1.0 (heterogeneity inflates)', 'Pass': 0.7 <= var_mean <= 1.6})

# 9. Log-Premium Shape
print("\n9. PREMIUM DISTRIBUTION SHAPE")
print("-"*60)
skewness = log_premiums.skew()
kurtosis = log_premiums.kurtosis()
dagostino_stat, dagostino_p = normaltest(log_premiums.sample(min(5000, len(log_premiums)), random_state=42))
print(f"   Log-premium skewness: {skewness:.3f} (target |skew| < 1.5)")
print(f"   Log-premium kurtosis: {kurtosis:.3f}")
print(f"   D'Agostino-Pearson: stat={dagostino_stat:.2f}, p={dagostino_p:.4e}")
enhanced_results.append({'Test': 'Log-Premium Skewness', 'Statistic': f"{skewness:.3f}", 'P-value': '-', 'Interpretation': '|skew| < 1.5 (tariff-fixed TPO flat premium widens left mass)', 'Pass': abs(skewness) < 1.5})

# Summary
print("\n" + "="*70)
print("ENHANCED VALIDATION SUMMARY")
print("="*70)
enhanced_df = pd.DataFrame(enhanced_results)
print("\n" + enhanced_df.to_string(index=False))
pass_count = enhanced_df['Pass'].sum()
total_count = len(enhanced_df)
print(f"\nResult: {pass_count}/{total_count} tests passed")
if pass_count == total_count:
    print("ALL ENHANCED VALIDATION TESTS PASSED")
else:
    print(f"{total_count - pass_count} test(s) failed - review above")
print("="*70)


DATASET VALIDATION
  Region premium ratio (East/Peninsular): 0.70x (report only)

                                     Test                      Value                                             Expected  Pass
Premium vs Sum Insured Correlation (Comp)                      0.795                                                > 0.5  True
             Young Driver Premium Loading                      1.18x > 1.05x (driver loading partly offset by newer cars)  True
                  Overall Claim Frequency                     13.87%                                               10-20%  True
            NCD Progression (2026 < 2030)             24.8% to 34.8%                                           Increasing  True
                       Overall Loss Ratio                     69.01%                                               50-80%  True
              Comprehensive Premium > TPO                     13.59x                                               > 1.8x  True
          TPFT Premium

In [20]:
# ============================================================================
# EV TREND MARKET ANALYSIS - descriptive EV/ICE split + adoption scenarios
# Theory (fair-value): premium tracks SA, own-risk severity tracks SA, TPBI/TPPD
# are SA-independent -> EV LR ~ ICE LR (slightly lower) as EV share rises.
# ============================================================================

# ---- 1. Descriptive: EV vs ICE by year (base run) ----
_res = cohort_results
_is_ent = _res['POLID'].str.startswith('ENT')

ev_all = _res.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
ev_ent = _res[_is_ent].groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())


def _ev_ice_metrics(g):
    out = {}
    for name, sub in [('EV', g[g['VEHICLE_TYPE'] == 'EV']),
                      ('ICE', g[g['VEHICLE_TYPE'] == 'ICE'])]:
        prem = sub['FINAL_PREMIUM_SST'].sum()
        ncl = sub['CLAIM_COUNT'].sum()
        out[name + '_lr'] = sub['CLAIM_AMOUNT'].sum() / prem if prem > 0 else float('nan')
        out[name + '_freq'] = sub['CLAIM_OCCURRED'].mean()
        out[name + '_sev'] = (sub['CLAIM_AMOUNT'].sum() / ncl) if ncl > 0 else float('nan')
        out[name + '_prem'] = sub['FINAL_PREMIUM_SST'].mean()
        out[name + '_ret'] = sub['RENEWED'].mean()
        out[name + '_ncd'] = sub['NCD_LEVEL'].mean()
    return pd.Series(out)


ev_ice = _res.groupby('SIM_YEAR').apply(_ev_ice_metrics)

print('=== EV adoption (base run) ===')
print('  EV share overall : {:.1%} -> {:.1%}'.format(ev_all.iloc[0], ev_all.iloc[-1]))
print('  EV share entrants: {:.1%} -> {:.1%}'.format(ev_ent.iloc[0], ev_ent.iloc[-1]))
print('\n=== EV vs ICE by year (base run) ===')
print(ev_ice.round(4).to_string())
print('\n  Final-year EV LR vs ICE LR: {:.1%} vs {:.1%}'.format(
    ev_ice['EV_lr'].iloc[-1], ev_ice['ICE_lr'].iloc[-1]))

# ---- 2. Adoption scenarios (same book + seed, only EV ramp varies) ----
_SCENARIOS = {
    'Conservative': {2026: 0.03, 2030: 0.06, 2035: 0.12, 2040: 0.18, 2045: 0.25},
    'Baseline':     dict(COHORT_CONFIG['ev_share_by_year']),
    'Aggressive':   {2026: 0.08, 2030: 0.20, 2035: 0.40, 2040: 0.65, 2045: 0.85},
}

_scen_rows = []
_scen_curves = {}
for _name, _sched in _SCENARIOS.items():
    _cfg = deep_update(COHORT_CONFIG, {'ev_share_by_year': _sched})
    _sim = simulate_cohort(df, n_years=20, new_entrants_per_year=None,
                           seed=42, cfg=_cfg, verbose=False)
    _final = _sim[_sim['SIM_YEAR'] == _sim['SIM_YEAR'].max()]
    _scen_rows.append({
        'scenario': _name,
        'final_ev_share': (_final['VEHICLE_TYPE'] == 'EV').mean(),
        'overall_lr': _sim['CLAIM_AMOUNT'].sum() / _sim['FINAL_PREMIUM_SST'].sum(),
        'n_policy_years': len(_sim),
    })
    _scen_curves[_name] = _sim.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(
        lambda s: (s == 'EV').mean())
scen_df = pd.DataFrame(_scen_rows)

print('\n=== EV adoption scenarios (same book, seed 42) ===')
print(scen_df.round(4).to_string(index=False))
lr_ord = scen_df.sort_values('final_ev_share')['overall_lr']
print('  LR ordering (fair-value: more EV -> slightly lower LR):',
      'OK' if list(lr_ord) == sorted(lr_ord, reverse=True) else 'CHECK')

# ---- 3. Charts ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(ev_all.index, ev_all.values, marker='o', label='Overall')
ax.plot(ev_ent.index, ev_ent.values, marker='s', label='Entrants')
ax.set_title('EV share over time (base run)')
ax.set_xlabel('Year')
ax.set_ylabel('EV share')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(ev_ice.index, ev_ice['EV_lr'].values, marker='o', label='EV')
ax.plot(ev_ice.index, ev_ice['ICE_lr'].values, marker='s', label='ICE')
ax.set_title('Loss ratio by vehicle type (base run)')
ax.set_xlabel('Year')
ax.set_ylabel('Loss ratio')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[2]
for _name, _curve in _scen_curves.items():
    _lr = scen_df.loc[scen_df['scenario'] == _name, 'overall_lr'].iloc[0]
    ax.plot(_curve.index, _curve.values, marker='o',
            label=_name + ' (LR {:.1%})'.format(_lr))
ax.set_title('EV share by scenario')
ax.set_xlabel('Year')
ax.set_ylabel('EV share')
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.savefig('images/ev_analysis.png', dpi=150)
print('\nChart saved: data/ev_analysis.png')


=== EV adoption (base run) ===
  EV share overall : 10.1% -> 10.0%
  EV share entrants: 9.8% -> 10.0%

=== EV vs ICE by year (base run) ===
           EV_lr  EV_freq      EV_sev    EV_prem  EV_ret  EV_ncd  ICE_lr  ICE_freq    ICE_sev   ICE_prem  ICE_ret  ICE_ncd
SIM_YEAR                                                                                                                  
2026      0.5619   0.1620   7037.7081  2203.8576  0.8101  0.3069  0.6119    0.1428  5802.4853  1480.3509   0.8210   0.3148
2027      0.5712   0.1607   6690.3288  2087.9939  0.7345  0.3330  0.6790    0.1436  5992.8531  1389.2964   0.7412   0.3415
2028      0.7432   0.1646   8486.4132  2036.1560  0.7401  0.3461  0.7346    0.1384  6474.8629  1348.3508   0.7535   0.3588
2029      0.5474   0.1492   7103.4164  2043.5633  0.6985  0.3573  0.6695    0.1405  5787.6446  1328.9586   0.7074   0.3660
2030      0.6626   0.1695   7329.9687  2041.6920  0.6780  0.3550  0.7613    0.1408  6491.0406  1314.7087   0.7098   0.3703

In [21]:
# ============================================================================
# OVERTHINKER-STYLE EDA (adapted from reference Section 7)
# ============================================================================

print("="*70)
print("GENERATING ACTUARIAL EDA REPORTS")
print("="*70)

# 1. Actuarial Premium Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pivot_agencd = final_dataset.groupby(['age_band', 'NCD_TIER'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
sns.heatmap(pivot_agencd, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'Median Premium (RM)'})
axes[0].set_title('Median Premium: Age Band vs NCD Tier')
axes[0].set_xlabel('NCD Tier')
axes[0].set_ylabel('Driver Age Band')

pivot_covloc = final_dataset.groupby(['COVERAGE_TYPE', 'REGION'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
pivot_covloc.columns = [c.replace(' Malaysia (Sabah, Sawarak & Labuan)', '').replace(' Malaysia', '') for c in pivot_covloc.columns]
pivot_covloc = pivot_covloc.reindex(['Comprehensive', 'TPFT', 'TPO'])
sns.heatmap(pivot_covloc, annot=True, fmt='.0f', cmap='Blues', ax=axes[1],
            cbar_kws={'label': 'Median Premium (RM)'})
axes[1].set_title('Median Premium: Coverage vs Region')
plt.tight_layout()
plt.savefig('images/eda_premium_heatmaps.png', dpi=150, bbox_inches='tight')
plt.close()

# 2. Risk Profile: Frequency & Loss Ratio by Age Band
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
freq_by_age = final_dataset.groupby('age_band', observed=True)['CLAIM_COUNT'].apply(lambda x: (x > 0).mean() * 100)
sns.barplot(x=freq_by_age.index, y=freq_by_age.values, ax=axes[0], palette='viridis')
axes[0].axhline((final_dataset['CLAIM_COUNT'] > 0).mean() * 100, ls='--', color='red', label='Portfolio Average')
axes[0].set_title('Claim Frequency by Age Band (U-Shaped Risk)')
axes[0].set_ylabel('Claim Frequency (%)')
axes[0].legend()

lr_by_age = final_dataset.groupby('age_band', observed=True).apply(
    lambda x: (x['CLAIM_AMOUNT'].sum() / x['FINAL_PREMIUM_SST'].sum()) * 100
)
sns.barplot(x=lr_by_age.index, y=lr_by_age.values, ax=axes[1], palette='magma')
axes[1].axhline((final_dataset['CLAIM_AMOUNT'].sum() / final_dataset['FINAL_PREMIUM_SST'].sum()) * 100,
                ls='--', color='red', label='Portfolio Average')
axes[1].set_title('Loss Ratio by Age Band')
axes[1].set_ylabel('Loss Ratio (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig('images/eda_risk_profile.png', dpi=150, bbox_inches='tight')
plt.close()

# 3. Severity Tails (Log-Scale)
plt.figure(figsize=(10, 6))
claimants = final_dataset[final_dataset['CLAIM_AMOUNT'] > 0]
sns.histplot(data=claimants, x=np.log1p(claimants['CLAIM_AMOUNT']),
             hue='COVERAGE_TYPE', hue_order=['Comprehensive', 'TPFT', 'TPO'],
             bins=50, kde=True, palette='Set2', alpha=0.6, element='step')
plt.title('Log-Scale Claim Severity Distribution (Fat Tails)')
plt.xlabel('log(1 + Claim Amount)')
plt.ylabel('Count of Claims')
plt.savefig('images/eda_severity_tails.png', dpi=150, bbox_inches='tight')
plt.close()

# 4. Severity Distributions by Coverage (Gamma overlay)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, cov in enumerate(['Comprehensive', 'TPFT', 'TPO']):
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 10:
        claims_subset.hist(bins=50, ax=axes[i], color=['skyblue', 'coral', 'seagreen'][i],
                           edgecolor='black', alpha=0.7, density=True)
        shape_f, loc_f, scale_f = gamma_dist.fit(claims_subset, floc=0)
        x = np.linspace(0, claims_subset.quantile(0.99), 200)
        axes[i].plot(x, gamma_dist.pdf(x, shape_f, loc_f, scale_f), 'r-', lw=2,
                     label=f'Gamma fit (k={shape_f:.1f})')
        axes[i].set_title(f'{cov} Severity Distribution', fontsize=13, fontweight='bold')
        axes[i].set_xlabel('Claim Amount (RM)')
        axes[i].legend()
        axes[i].axvline(claims_subset.mean(), color='black', linestyle='--', alpha=0.5, label='Mean')
        axes[i].axvline(claims_subset.quantile(0.95), color='red', linestyle=':', alpha=0.5, label='P95')
plt.tight_layout()
plt.savefig('images/severity_distributions.png', dpi=150, bbox_inches='tight')
plt.close()

# 5. Longitudinal Trends
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
year_stats = final_dataset.groupby('SIM_YEAR').agg({
    'FINAL_PREMIUM_SST': 'mean',
    'NCD_LEVEL_PRICED': 'mean',
    'CLAIM_COUNT': lambda x: (x > 0).mean()
}).reset_index()
axes[0].plot(year_stats['SIM_YEAR'], year_stats['FINAL_PREMIUM_SST'], 'b-o', linewidth=2, markersize=8)
axes[0].set_title('Average Premium by Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('SIM_YEAR')
axes[0].set_ylabel('Avg Premium (RM)')
axes[0].grid(True, alpha=0.3)
ax2 = axes[0].twinx()
ax2.plot(year_stats['SIM_YEAR'], year_stats['NCD_LEVEL_PRICED']*100, 'r--s', linewidth=2, markersize=6, alpha=0.7)
ax2.set_ylabel('Avg NCD %', color='red')

for label, age_min, age_max, color in [('Gen Z (18-27)', 18, 27, 'red'),
                                       ('Prime (28-50)', 28, 50, 'blue'),
                                       ('Senior (51+)', 51, 100, 'green')]:
    subset = final_dataset[(final_dataset['DRIVER_AGE'] >= age_min) & (final_dataset['DRIVER_AGE'] <= age_max)]
    rates = subset.groupby('SIM_YEAR')['CLAIM_COUNT'].apply(lambda x: (x > 0).mean() * 100)
    axes[1].plot(rates.index, rates.values, '-o', label=label, color=color, linewidth=2)
axes[1].set_title('Claim Rate by Year & Age Group', fontsize=14, fontweight='bold')
axes[1].set_xlabel('SIM_YEAR')
axes[1].set_ylabel('Claim Rate (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/longitudinal_trends.png', dpi=150, bbox_inches='tight')
plt.close()

print("EDA visualisations generated and saved:")
print("  images/correlation_heatmap.png, images/ncd_validation.png")
print("  images/eda_premium_heatmaps.png, images/eda_risk_profile.png")
print("  images/eda_severity_tails.png, images/severity_distributions.png")
print("  images/longitudinal_trends.png")


GENERATING ACTUARIAL EDA REPORTS


C:\Users\Admin\AppData\Local\Temp\ipykernel_1840\3679051989.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=freq_by_age.index, y=freq_by_age.values, ax=axes[0], palette='viridis')
C:\Users\Admin\AppData\Local\Temp\ipykernel_1840\3679051989.py:40: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=lr_by_age.index, y=lr_by_age.values, ax=axes[1], palette='magma')


EDA visualisations generated and saved:
  images/correlation_heatmap.png, images/ncd_validation.png
  images/eda_premium_heatmaps.png, images/eda_risk_profile.png
  images/eda_severity_tails.png, images/severity_distributions.png
  images/longitudinal_trends.png


In [22]:
# ============================================================================
# EXPORT SIMULATION DATA TO CSV (data/)
# Reproducible: deterministic (seed 42), regenerated on every execution
# ============================================================================


os.makedirs('data', exist_ok=True)

cohort_results.to_csv('data/simulation_cohort_results.csv', index=False)
df.to_csv('data/initial_book.csv', index=False)

print('Exports written to data/:')
for f in sorted(os.listdir('data')):
    p = os.path.join('data', f)
    print(f'  {f:38s} {os.path.getsize(p):>12,} bytes')
print(f'cohort_results: {len(cohort_results):,} rows x {cohort_results.shape[1]} cols')
print(f'initial book  : {len(df):,} rows x {df.shape[1]} cols')

Exports written to data/:
  initial_book.csv                          1,950,812 bytes
  model_parameters.csv                          1,295 bytes
  simulation_cohort_results.csv            67,677,553 bytes
cohort_results: 328,105 rows x 27 cols
initial book  : 10,000 rows x 20 cols
